# WeltmeisterKI4 Inference Notebook

Dieses Notebook ist die schlanke Nutzer-Version: Es trainiert nichts neu, sondern lädt die fertigen Modellfiles, rekonstruiert den historischen Feature-State, übernimmt bereits gespielte WM-Spiele und erzeugt danach Prognosen, Einzelbaum und Monte-Carlo-Auswertungen.

Lege dieses Notebook in denselben Projektordner wie teams.csv, matches.csv, tournament_stages.csv, host_cities.csv und den models/-Ordner.


## 02. Setup und Daten laden


In [ ]:
# Zelle 03: Code
# Was diese Zelle macht:
# Setzt Projektpfade relativ zum Notebook, lädt historische Basisdaten und findet den Modellordner. Kein Training.

import os, sys, shutil, urllib.request, json, time, re, math, unicodedata
from pathlib import Path
from datetime import date, datetime, timedelta
from collections import defaultdict, deque, Counter

import numpy as np
import polars as pl

PROJECT = Path.cwd().resolve()
# Falls Jupyter aus einem anderen Ordner startet, hier manuell setzen:
# PROJECT = Path(r"D:/ml/projects/WeltmeisterKI").resolve()

DATA = PROJECT / "data"
RAW = DATA / "raw"
PROCESSED = DATA / "processed"
MODEL_CANDIDATES = [PROJECT / "models", PROJECT]
MODELS = next((p for p in MODEL_CANDIDATES if (p / "xgb_goal_models_v2_ensemble.joblib").exists()), PROJECT / "models")
SB_RAW = RAW / "statsbomb"
FIFA_RAW = RAW / "fifa"
TMP_DIR = PROJECT / ".tmp"

for p in [DATA, RAW, PROCESSED, MODELS, SB_RAW, FIFA_RAW, TMP_DIR]:
    p.mkdir(parents=True, exist_ok=True)

os.environ["TMP"] = str(TMP_DIR)
os.environ["TEMP"] = str(TMP_DIR)
os.environ["PIP_CACHE_DIR"] = str(TMP_DIR / "pip_cache")

print("PROJECT:", PROJECT)
print("MODELS:", MODELS)
print("RAW:", RAW)
print("PROCESSED:", PROCESSED)

URLS = {
    "results": "https://raw.githubusercontent.com/martj42/international_results/master/results.csv",
    "goalscorers": "https://raw.githubusercontent.com/martj42/international_results/master/goalscorers.csv",
    "shootouts": "https://raw.githubusercontent.com/martj42/international_results/master/shootouts.csv",
}

def download(url, out):
    out = Path(out)
    if not out.exists() or out.stat().st_size == 0:
        print("download", out.name)
        urllib.request.urlretrieve(url, out)
    return out

for name, url in URLS.items():
    download(url, RAW / f"{name}.csv")

NULLS = ["", "NA", "N/A", "NULL", "null", "None"]

results_raw = pl.read_csv(
    RAW / "results.csv",
    null_values=NULLS,
    infer_schema_length=20000,
    schema_overrides={"home_score": pl.Int64, "away_score": pl.Int64, "neutral": pl.Boolean},
    try_parse_dates=True,
)

results = (
    results_raw
    .filter(pl.col("date") >= pl.date(1980, 1, 1))
    .filter(pl.col("home_score").is_not_null() & pl.col("away_score").is_not_null())
    .sort("date")
)

goalscorers = pl.read_csv(RAW / "goalscorers.csv", null_values=NULLS, infer_schema_length=20000, try_parse_dates=True)
shootouts = pl.read_csv(RAW / "shootouts.csv", null_values=NULLS, infer_schema_length=20000, try_parse_dates=True)

required_models = [
    MODELS / "xgb_goal_models_v2_ensemble.joblib",
    MODELS / "weltmeisterki_deep_v1.pt",
    MODELS / "weltmeisterki4_sota_meta.joblib",
]
missing_models = [str(p) for p in required_models if not p.exists()]
if missing_models:
    missing_text = "\\n".join(missing_models)
    raise FileNotFoundError(
        "Diese Inference-Version braucht die trainierten Modellfiles im models/-Ordner oder direkt neben dem Notebook. Fehlt:\\n"
        + missing_text
        + "\\n\\nBitte einmal das SOTA-Trainingsnotebook ausfuehren oder die Modellfiles in diesen Ordner kopieren."
    )

print("results:", results.shape)
print("goalscorers:", goalscorers.shape)
print("shootouts:", shootouts.shape)
print("model files ok")


## 04. FIFA-Ranking laden


In [ ]:
# Zelle 05: Code
# Was diese Zelle macht:
# Lädt FIFA-Ranking-Daten, wenn sie lokal vorhanden sind. Falls nicht, laufen FIFA-Features als Missing Values weiter.

def norm_team(s):
    if s is None:
        return ""
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode("ascii")
    s = re.sub(r"s+", " ", s).strip()
    aliases = {
        "United States": "USA",
        "Korea Republic": "South Korea",
        "Republic of Korea": "South Korea",
        "Cote dIvoire": "Ivory Coast",
        "Cote d'Ivoire": "Ivory Coast",
        "Curaçao": "Curacao",
        "Curacao": "Curacao",
        "Iran": "IR Iran",
        "DR Congo": "DR Congo",
    }
    return aliases.get(s, s)

def norm_col(c):
    return re.sub(r"[^a-z0-9]+", "_", str(c).strip().lower()).strip("_")

def read_fifa_csv(path):
    try:
        df = pl.read_csv(path, infer_schema=False, null_values=NULLS, ignore_errors=True)
    except Exception:
        return None
    df = df.rename({c: norm_col(c) for c in df.columns})
    cols = set(df.columns)
    date_col = next((c for c in ["rank_date", "date", "ranking_date"] if c in cols), None)
    country_col = next((c for c in ["country_full", "country", "team", "name"] if c in cols), None)
    rank_col = next((c for c in ["rank", "ranking"] if c in cols), None)
    points_col = next((c for c in ["total_points", "points", "previous_points"] if c in cols), None)
    conf_col = next((c for c in ["confederation", "confed"] if c in cols), None)
    if not date_col or not country_col or not rank_col:
        return None
    out = df.select([
        pl.col(date_col).str.to_date(strict=False).alias("rank_date"),
        pl.col(country_col).cast(pl.String).alias("country_full"),
        pl.col(rank_col).cast(pl.Float64, strict=False).alias("rank"),
        (pl.col(points_col).cast(pl.Float64, strict=False) if points_col else pl.lit(float("nan"))).alias("total_points"),
        (pl.col(conf_col).cast(pl.String) if conf_col else pl.lit("")).alias("confederation"),
    ])
    return out.drop_nulls(["rank_date", "country_full", "rank"])

fifa_path = PROCESSED / "fifa_rankings.parquet"
if fifa_path.exists():
    fifa_rankings = pl.read_parquet(fifa_path)
else:
    parts = []
    for f in FIFA_RAW.glob("*.csv"):
        part = read_fifa_csv(f)
        if part is not None and part.height:
            parts.append(part)
    if parts:
        fifa_rankings = (
            pl.concat(parts, how="vertical_relaxed")
            .unique(subset=["rank_date", "country_full"], keep="last")
            .sort(["rank_date", "rank"])
        )
        fifa_rankings.write_parquet(fifa_path)
    else:
        fifa_rankings = pl.DataFrame({
            "rank_date": [], "country_full": [], "rank": [], "total_points": [], "confederation": []
        }, schema={
            "rank_date": pl.Date, "country_full": pl.String, "rank": pl.Float64,
            "total_points": pl.Float64, "confederation": pl.String
        })
        print("WARNUNG: Keine FIFA-Rankings gefunden. FIFA-Features werden als Missing Values imputiert.")

rank_hist = defaultdict(list)
for r in fifa_rankings.iter_rows(named=True):
    rank_hist[norm_team(r["country_full"])].append((
        r["rank_date"],
        float(r["rank"]),
        float(r["total_points"]) if r.get("total_points") is not None else math.nan,
        str(r.get("confederation") or ""),
    ))
for t in rank_hist:
    rank_hist[t].sort(key=lambda x: x[0])

def fifa_before(team_norm, d):
    arr = rank_hist.get(team_norm, [])
    if not arr:
        return math.nan, math.nan, ""
    lo, hi = 0, len(arr)
    while lo < hi:
        mid = (lo + hi) // 2
        if arr[mid][0] < d:
            lo = mid + 1
        else:
            hi = mid
    if lo == 0:
        return math.nan, math.nan, ""
    _, rank, points, conf = arr[lo - 1]
    return rank, points, conf

print("FIFA rows:", fifa_rankings.height)
print("Teams with FIFA history:", len(rank_hist))
if fifa_rankings.height:
    print("FIFA range:", fifa_rankings["rank_date"].min(), fifa_rankings["rank_date"].max())


## 06. Zusatz-Lookups


In [ ]:
# Zelle 07: Code
# Was diese Zelle macht:
# Baut Zusatz-Lookups aus Torschützen und Shootouts. StatsBomb-xG ist optional; ohne lokale Daten bleibt sb_map leer.

def empty_goal_bucket():
    return {"scorers": set(), "pen": 0, "own": 0}

goal_map = defaultdict(lambda: defaultdict(empty_goal_bucket))
if goalscorers.height:
    for r in goalscorers.iter_rows(named=True):
        d = r.get("date")
        h = r.get("home_team")
        a = r.get("away_team")
        team = r.get("team")
        scorer = r.get("scorer")
        if d is None or h is None or a is None or team is None:
            continue
        bucket = goal_map[(d, h, a)][team]
        if scorer is not None:
            bucket["scorers"].add(str(scorer))
        if bool(r.get("penalty") or False):
            bucket["pen"] += 1
        if bool(r.get("own_goal") or False):
            bucket["own"] += 1

shootout_map = {}
if shootouts.height:
    for r in shootouts.iter_rows(named=True):
        d = r.get("date")
        h = r.get("home_team")
        a = r.get("away_team")
        w = r.get("winner")
        if d is not None and h is not None and a is not None and w is not None:
            shootout_map[(d, h, a)] = w

sb_map = {}
sb_features_path = PROCESSED / "statsbomb_match_features.parquet"
if sb_features_path.exists():
    try:
        sb_df = pl.read_parquet(sb_features_path)
        for r in sb_df.iter_rows(named=True):
            sb_map[(r["date"], norm_team(r["team"]), norm_team(r["opponent"]))] = r
        print("StatsBomb features loaded:", len(sb_map))
    except Exception as e:
        print("StatsBomb optional load failed:", repr(e))
else:
    print("StatsBomb optional: keine lokalen Features gefunden, sb_map bleibt leer.")

print("goal_map matches:", len(goal_map))
print("shootouts:", len(shootout_map))


## 08. Historischen Feature-State rekonstruieren


In [ ]:
# Zelle 09: Code
# Was diese Zelle macht:
# Baut die zentrale Feature-Tabelle ohne Data Leakage: Elo, FIFA, Form, H2H, xG, Scorer, Shootouts sowie Attack/Defense-Ratings. Ergebnis: features und state für spätere Fixtures.

from collections import defaultdict, deque
import math
import numpy as np
import polars as pl
from datetime import date

BASE_GOALS_PER_TEAM = 1.35
ATT_DEF_ALPHA = 0.055

if "sb_map" not in globals():
    sb_map = {}

if "shootout_map" not in globals():
    shootout_map = {}

def safe_div(a, b):
    return float(a / b) if b else np.nan

def result_points(gf, ga):
    if gf > ga:
        return 1.0
    if gf == ga:
        return 0.5
    return 0.0

def elo_expected(a, b):
    return 1.0 / (1.0 + 10 ** ((b - a) / 400.0))

def seq_avg(seq, fn, n):
    xs = [fn(x) for x in list(seq)[-n:]]
    xs = [x for x in xs if x is not None and not (isinstance(x, float) and math.isnan(x))]
    return float(np.mean(xs)) if xs else np.nan

def days_since_last(seq, current_date):
    if not seq:
        return np.nan
    return float((current_date - seq[-1]["date"]).days)

def tournament_weight(t):
    s = str(t).lower()
    if s == "fifa world cup":
        return 3.0
    if "qualification" in s:
        return 1.35
    if any(x in s for x in ["uefa euro", "copa america", "african cup", "asian cup", "gold cup", "nations league"]):
        return 2.0
    if s == "friendly":
        return 0.8
    return 1.0

def empty_goal_stats():
    return {"scorers": set(), "pen": 0, "own": 0}

def get_goal_stats(d, h, a, team):
    try:
        return goal_map[(d, h, a)][team]
    except Exception:
        return empty_goal_stats()

def init_state_v2():
    return {
        "elo": defaultdict(lambda: 1500.0),
        "attack": defaultdict(lambda: 1.0),
        "def_weak": defaultdict(lambda: 1.0),
        "team_hist": defaultdict(lambda: deque(maxlen=60)),
        "xg_hist": defaultdict(lambda: deque(maxlen=30)),
        "scorer_hist": defaultdict(lambda: deque(maxlen=30)),
        "h2h": defaultdict(lambda: deque(maxlen=25)),
        "shoot_w": defaultdict(int),
        "shoot_l": defaultdict(int),
    }

def snapshot_features_v2(home, away, d, tournament="FIFA World Cup", country="United States", neutral=True, state=None):
    hn, an = norm_team(home), norm_team(away)

    elo = state["elo"]
    attack = state["attack"]
    def_weak = state["def_weak"]
    team_hist = state["team_hist"]
    xg_hist = state["xg_hist"]
    scorer_hist = state["scorer_hist"]
    h2h = state["h2h"]
    shoot_w = state["shoot_w"]
    shoot_l = state["shoot_l"]

    hh, ah = team_hist[hn], team_hist[an]
    hx, ax = xg_hist[hn], xg_hist[an]
    hsc, asc = scorer_hist[hn], scorer_hist[an]
    h2h_ha = h2h[(hn, an)]

    h_rank, h_rank_pts, h_conf = fifa_before(hn, d)
    a_rank, a_rank_pts, a_conf = fifa_before(an, d)

    row = {
        "date": d,
        "home_team": home,
        "away_team": away,

        "year": d.year,
        "month": d.month,
        "days_from_1980": (d - date(1980, 1, 1)).days,

        "neutral": int(bool(neutral)),
        "home_country_host": int(norm_team(country) == hn and not neutral),
        "away_country_host": int(norm_team(country) == an and not neutral),

        "is_world_cup": int(tournament == "FIFA World Cup"),
        "is_qualifier": int("qualification" in str(tournament).lower()),
        "is_friendly": int(str(tournament).lower() == "friendly"),
        "is_major_tournament": int(any(x in str(tournament).lower() for x in [
            "fifa world cup", "uefa euro", "copa america", "african cup",
            "asian cup", "gold cup", "nations league"
        ])),

        "elo_home_pre": elo[hn],
        "elo_away_pre": elo[an],
        "elo_diff": elo[hn] - elo[an],
        "elo_ratio": safe_div(elo[hn], elo[an]),

        "home_attack_pre": attack[hn],
        "away_attack_pre": attack[an],
        "home_def_weak_pre": def_weak[hn],
        "away_def_weak_pre": def_weak[an],
        "attack_diff": attack[hn] - attack[an],
        "def_weak_diff": def_weak[hn] - def_weak[an],
        "home_attack_vs_away_def": attack[hn] * def_weak[an],
        "away_attack_vs_home_def": attack[an] * def_weak[hn],

        "home_fifa_rank": h_rank,
        "away_fifa_rank": a_rank,
        "fifa_rank_diff": h_rank - a_rank if not math.isnan(h_rank) and not math.isnan(a_rank) else np.nan,
        "home_fifa_points": h_rank_pts,
        "away_fifa_points": a_rank_pts,
        "fifa_points_diff": h_rank_pts - a_rank_pts if not math.isnan(h_rank_pts) and not math.isnan(a_rank_pts) else np.nan,
        "same_confed": int(h_conf != "" and h_conf == a_conf),

        "home_games_pre": len(hh),
        "away_games_pre": len(ah),
        "home_days_rest": days_since_last(hh, d),
        "away_days_rest": days_since_last(ah, d),
        "rest_diff": days_since_last(hh, d) - days_since_last(ah, d) if hh and ah else np.nan,

        "home_shootout_wins_pre": shoot_w[hn],
        "away_shootout_wins_pre": shoot_w[an],
        "home_shootout_losses_pre": shoot_l[hn],
        "away_shootout_losses_pre": shoot_l[an],
    }

    for n in [3, 5, 10, 20]:
        for side, hist in [("home", hh), ("away", ah)]:
            row[f"{side}_gf_l{n}"] = seq_avg(hist, lambda x: x["gf"], n)
            row[f"{side}_ga_l{n}"] = seq_avg(hist, lambda x: x["ga"], n)
            row[f"{side}_gd_l{n}"] = seq_avg(hist, lambda x: x["gf"] - x["ga"], n)
            row[f"{side}_pts_l{n}"] = seq_avg(hist, lambda x: x["pts"], n)
            row[f"{side}_adj_gf_l{n}"] = seq_avg(hist, lambda x: x["adj_gf"], n)
            row[f"{side}_adj_ga_l{n}"] = seq_avg(hist, lambda x: x["adj_ga"], n)
            row[f"{side}_adj_gd_l{n}"] = seq_avg(hist, lambda x: x["adj_gf"] - x["adj_ga"], n)
            row[f"{side}_adj_pts_l{n}"] = seq_avg(hist, lambda x: x["adj_pts"], n)
            row[f"{side}_win_rate_l{n}"] = seq_avg(hist, lambda x: int(x["pts"] == 1.0), n)
            row[f"{side}_loss_rate_l{n}"] = seq_avg(hist, lambda x: int(x["pts"] == 0.0), n)
            row[f"{side}_clean_sheet_l{n}"] = seq_avg(hist, lambda x: int(x["ga"] == 0), n)
            row[f"{side}_failed_score_l{n}"] = seq_avg(hist, lambda x: int(x["gf"] == 0), n)

        row[f"form_pts_diff_l{n}"] = row[f"home_pts_l{n}"] - row[f"away_pts_l{n}"] if not math.isnan(row[f"home_pts_l{n}"]) and not math.isnan(row[f"away_pts_l{n}"]) else np.nan
        row[f"form_gd_diff_l{n}"] = row[f"home_gd_l{n}"] - row[f"away_gd_l{n}"] if not math.isnan(row[f"home_gd_l{n}"]) and not math.isnan(row[f"away_gd_l{n}"]) else np.nan
        row[f"adj_form_pts_diff_l{n}"] = row[f"home_adj_pts_l{n}"] - row[f"away_adj_pts_l{n}"] if not math.isnan(row[f"home_adj_pts_l{n}"]) and not math.isnan(row[f"away_adj_pts_l{n}"]) else np.nan
        row[f"adj_form_gd_diff_l{n}"] = row[f"home_adj_gd_l{n}"] - row[f"away_adj_gd_l{n}"] if not math.isnan(row[f"home_adj_gd_l{n}"]) and not math.isnan(row[f"away_adj_gd_l{n}"]) else np.nan

    for n in [3, 5, 10]:
        for side, hist in [("home", hx), ("away", ax)]:
            row[f"{side}_xg_for_l{n}"] = seq_avg(hist, lambda x: x["xg_for"], n)
            row[f"{side}_xg_against_l{n}"] = seq_avg(hist, lambda x: x["xg_against"], n)
            row[f"{side}_xg_diff_l{n}"] = seq_avg(hist, lambda x: x["xg_for"] - x["xg_against"], n)
            row[f"{side}_shots_l{n}"] = seq_avg(hist, lambda x: x["shots"], n)
            row[f"{side}_sot_l{n}"] = seq_avg(hist, lambda x: x["sot"], n)
            row[f"{side}_passes_l{n}"] = seq_avg(hist, lambda x: x["passes"], n)
            row[f"{side}_pressures_l{n}"] = seq_avg(hist, lambda x: x["pressures"], n)

    for n in [5, 10, 20]:
        row[f"home_unique_scorers_l{n}"] = seq_avg(hsc, lambda x: x["unique_scorers"], n)
        row[f"away_unique_scorers_l{n}"] = seq_avg(asc, lambda x: x["unique_scorers"], n)
        row[f"home_pen_goals_l{n}"] = seq_avg(hsc, lambda x: x["pen"], n)
        row[f"away_pen_goals_l{n}"] = seq_avg(asc, lambda x: x["pen"], n)
        row[f"home_own_goals_l{n}"] = seq_avg(hsc, lambda x: x["own"], n)
        row[f"away_own_goals_l{n}"] = seq_avg(asc, lambda x: x["own"], n)

    for n in [3, 5, 10]:
        row[f"h2h_home_pts_l{n}"] = seq_avg(h2h_ha, lambda x: x["pts"], n)
        row[f"h2h_home_gd_l{n}"] = seq_avg(h2h_ha, lambda x: x["gd"], n)
        row[f"h2h_games_l{n}"] = min(len(h2h_ha), n)

    row["sample_weight"] = tournament_weight(tournament) * (0.75 + (d.year - 1980) / 60.0)

    return row

def update_state_after_match_v2(r, state):
    d = r["date"]
    h = r["home_team"]
    a = r["away_team"]
    hn = norm_team(h)
    an = norm_team(a)
    hs = int(r["home_score"])
    aw = int(r["away_score"])
    t = r["tournament"]

    elo = state["elo"]
    attack = state["attack"]
    def_weak = state["def_weak"]
    team_hist = state["team_hist"]
    xg_hist = state["xg_hist"]
    scorer_hist = state["scorer_hist"]
    h2h = state["h2h"]
    shoot_w = state["shoot_w"]
    shoot_l = state["shoot_l"]

    h_elo_pre = elo[hn]
    a_elo_pre = elo[an]

    hp = result_points(hs, aw)
    ap = result_points(aw, hs)

    h_adj_gf = hs * (a_elo_pre / 1500.0)
    h_adj_ga = aw * (1500.0 / max(1.0, a_elo_pre))
    a_adj_gf = aw * (h_elo_pre / 1500.0)
    a_adj_ga = hs * (1500.0 / max(1.0, h_elo_pre))

    team_hist[hn].append({
        "date": d, "gf": hs, "ga": aw, "pts": hp,
        "adj_gf": h_adj_gf, "adj_ga": h_adj_ga, "adj_pts": hp * (a_elo_pre / 1500.0),
    })
    team_hist[an].append({
        "date": d, "gf": aw, "ga": hs, "pts": ap,
        "adj_gf": a_adj_gf, "adj_ga": a_adj_ga, "adj_pts": ap * (h_elo_pre / 1500.0),
    })

    h_sb = sb_map.get((d, hn, an))
    a_sb = sb_map.get((d, an, hn))
    if h_sb and a_sb:
        xg_hist[hn].append({"xg_for": h_sb["sb_xg_for"], "xg_against": a_sb["sb_xg_for"], "shots": h_sb["sb_shots_for"], "sot": h_sb["sb_sot_for"], "passes": h_sb["sb_passes_for"], "pressures": h_sb["sb_pressures_for"]})
        xg_hist[an].append({"xg_for": a_sb["sb_xg_for"], "xg_against": h_sb["sb_xg_for"], "shots": a_sb["sb_shots_for"], "sot": a_sb["sb_sot_for"], "passes": a_sb["sb_passes_for"], "pressures": a_sb["sb_pressures_for"]})

    gh = get_goal_stats(d, h, a, h)
    ga = get_goal_stats(d, h, a, a)
    scorer_hist[hn].append({"unique_scorers": len(gh["scorers"]), "pen": gh["pen"], "own": gh["own"]})
    scorer_hist[an].append({"unique_scorers": len(ga["scorers"]), "pen": ga["pen"], "own": ga["own"]})

    h2h[(hn, an)].append({"pts": hp, "gd": hs - aw})
    h2h[(an, hn)].append({"pts": ap, "gd": aw - hs})

    winner = shootout_map.get((d, h, a))
    if winner:
        wn = norm_team(winner)
        loser = an if wn == hn else hn
        shoot_w[wn] += 1
        shoot_l[loser] += 1

    attack[hn] = (1 - ATT_DEF_ALPHA) * attack[hn] + ATT_DEF_ALPHA * max(0.15, h_adj_gf / BASE_GOALS_PER_TEAM)
    attack[an] = (1 - ATT_DEF_ALPHA) * attack[an] + ATT_DEF_ALPHA * max(0.15, a_adj_gf / BASE_GOALS_PER_TEAM)
    def_weak[hn] = (1 - ATT_DEF_ALPHA) * def_weak[hn] + ATT_DEF_ALPHA * max(0.15, h_adj_ga / BASE_GOALS_PER_TEAM)
    def_weak[an] = (1 - ATT_DEF_ALPHA) * def_weak[an] + ATT_DEF_ALPHA * max(0.15, a_adj_ga / BASE_GOALS_PER_TEAM)

    k = 20 * tournament_weight(t) * (1 + abs(hs - aw) / 4)
    eh = elo_expected(elo[hn], elo[an])
    elo[hn] += k * (hp - eh)
    elo[an] += k * (ap - (1 - eh))

state = init_state_v2()
rows = []

for r in results.sort("date").iter_rows(named=True):
    row = snapshot_features_v2(
        r["home_team"], r["away_team"], r["date"],
        tournament=r["tournament"],
        country=r["country"],
        neutral=bool(r["neutral"]),
        state=state,
    )
    row["home_score"] = int(r["home_score"])
    row["away_score"] = int(r["away_score"])
    rows.append(row)
    update_state_after_match_v2(r, state)

features = pl.DataFrame(rows).sort("date")
features.write_parquet(PROCESSED / "features_1980_plus_v2.parquet")

print("Features v2:", features.shape)
print("New feature examples:", [c for c in features.columns if "adj_" in c or "attack" in c or "def_weak" in c][:40])
features.head()


## 10. v2/XGBoost-Modell laden


In [ ]:
# Zelle 11: Code
# Was diese Zelle macht:
# Lädt v2 aus der Joblib-Datei und definiert predict_match/predict_neutral plus score_grid. Diese Namen sind die stabile Schnittstelle für Forecasts und Monte Carlo.

import joblib
import numpy as np
import math

bundle = joblib.load(MODELS / "xgb_goal_models_v2_ensemble.joblib")

feature_cols = bundle["feature_cols"]
imp = bundle["imputer"]
home_models = bundle["home_models"]
away_models = bundle["away_models"]
outcome_models = bundle["outcome_models"]
calibration = bundle["calibration"]

print("Loaded:", bundle["model_version"])
print("Features:", len(feature_cols))
print("Calibration:", calibration)

def predict_row_parts(row):
    x = np.array([[row.get(c, np.nan) for c in feature_cols]], dtype=float)
    x = imp.transform(x)

    home_xg = float(np.mean([m.predict(x)[0] for m in home_models]))
    away_xg = float(np.mean([m.predict(x)[0] for m in away_models]))

    home_xg = float(np.clip(home_xg, 0.03, 7.0))
    away_xg = float(np.clip(away_xg, 0.03, 7.0))

    clf_probs = np.mean([m.predict_proba(x)[0] for m in outcome_models], axis=0)
    clf_probs = np.clip(clf_probs, 1e-6, 1.0)
    clf_probs = clf_probs / clf_probs.sum()

    return home_xg, away_xg, clf_probs

def calibrated_score_grid(home_xg, away_xg, clf_probs=None, max_goals=10):
    lam_scale = calibration["lambda_scale"]
    draw_boost = calibration["draw_boost"]
    poisson_weight = calibration["poisson_weight"]

    home_xg = float(np.clip(home_xg, 0.03, 7.0))
    away_xg = float(np.clip(away_xg, 0.03, 7.0))

    hlam = float(np.clip(home_xg * lam_scale, 0.03, 7.0))
    alam = float(np.clip(away_xg * lam_scale, 0.03, 7.0))

    hp = np.array(
        [math.exp(-hlam) * hlam**k / math.factorial(k) for k in range(max_goals + 1)],
        dtype=float,
    )
    ap = np.array(
        [math.exp(-alam) * alam**k / math.factorial(k) for k in range(max_goals + 1)],
        dtype=float,
    )

    hp /= hp.sum()
    ap /= ap.sum()

    grid = np.outer(hp, ap)

    for i in range(max_goals + 1):
        grid[i, i] *= draw_boost

    grid /= grid.sum()

    p_home = float(np.tril(grid, -1).sum())
    p_draw = float(np.trace(grid))
    p_away = float(np.triu(grid, 1).sum())

    p_pois = np.array([p_home, p_draw, p_away], dtype=float)
    p_pois = np.clip(p_pois, 1e-9, 1.0)
    p_pois = p_pois / p_pois.sum()

    if clf_probs is None:
        final_outcome = p_pois
    else:
        clf_probs = np.asarray(clf_probs, dtype=float)
        clf_probs = np.clip(clf_probs, 1e-6, 1.0)
        clf_probs = clf_probs / clf_probs.sum()

        final_outcome = poisson_weight * p_pois + (1.0 - poisson_weight) * clf_probs
        final_outcome = np.clip(final_outcome, 1e-6, 1.0)
        final_outcome = final_outcome / final_outcome.sum()

        ratios = final_outcome / np.clip(p_pois, 1e-9, 1.0)

        for h in range(max_goals + 1):
            for a in range(max_goals + 1):
                if h > a:
                    grid[h, a] *= ratios[0]
                elif h == a:
                    grid[h, a] *= ratios[1]
                else:
                    grid[h, a] *= ratios[2]

        grid /= grid.sum()

    return grid, final_outcome

def raw_predict_home_away(home, away, match_date, tournament="FIFA World Cup", country="United States", neutral=True):
    """
    Kompatibel mit deiner alten Monte-Carlo-Zelle:
    Gibt weiterhin nur (home_xg, away_xg) zurück.
    """
    row = snapshot_features_v2(
        home,
        away,
        match_date,
        tournament=tournament,
        country=country,
        neutral=neutral,
        state=state,
    )

    home_xg, away_xg, clf_probs = predict_row_parts(row)
    return home_xg, away_xg

def predict_home_away(home, away, match_date, tournament="FIFA World Cup", country="United States", neutral=True, max_goals=10):
    """
    Neuer interner Helper, aber mit neutralem Namen.
    Gibt vollen Detail-Output für home vs away.
    """
    row = snapshot_features_v2(
        home,
        away,
        match_date,
        tournament=tournament,
        country=country,
        neutral=neutral,
        state=state,
    )

    home_xg, away_xg, clf_probs = predict_row_parts(row)
    grid, outcome_probs = calibrated_score_grid(
        home_xg,
        away_xg,
        clf_probs=clf_probs,
        max_goals=max_goals,
    )

    top = []
    for h in range(max_goals + 1):
        for a in range(max_goals + 1):
            top.append((h, a, float(grid[h, a])))

    top = sorted(top, key=lambda x: x[2], reverse=True)

    return {
        "home": home,
        "away": away,
        "home_xg": home_xg,
        "away_xg": away_xg,
        "p_home_win": float(outcome_probs[0]),
        "p_draw": float(outcome_probs[1]),
        "p_away_win": float(outcome_probs[2]),
        "most_likely_score": f"{top[0][0]}:{top[0][1]}",
        "most_likely_score_prob": top[0][2],
        "top_scorelines": top[:12],
        "score_grid": grid,
        "clf_probs": clf_probs,
    }

def predict_match(home, away, match_date, tournament="FIFA World Cup", country="United States", neutral=True, max_goals=10):
    """
    Alias für deine alte/naheliegende Schreibweise.
    """
    return predict_home_away(
        home,
        away,
        match_date,
        tournament=tournament,
        country=country,
        neutral=neutral,
        max_goals=max_goals,
    )

def predict_neutral(team, opponent, match_date, tournament="FIFA World Cup", country="United States", max_goals=10):
    """
    Kompatibel mit deinen bestehenden Vorhersage-Zellen.

    Deine alten Zellen erwarten Keys wie:
    - team_xg_pred
    - opponent_xg_pred
    - p_team_win
    - p_draw
    - p_opponent_win
    - most_likely_score
    - most_likely_score_prob
    - top_scorelines
    """
    p1 = predict_home_away(
        team,
        opponent,
        match_date,
        tournament=tournament,
        country=country,
        neutral=True,
        max_goals=max_goals,
    )

    p2 = predict_home_away(
        opponent,
        team,
        match_date,
        tournament=tournament,
        country=country,
        neutral=True,
        max_goals=max_goals,
    )

    team_xg = (p1["home_xg"] + p2["away_xg"]) / 2
    opp_xg = (p1["away_xg"] + p2["home_xg"]) / 2

    p_team_win = (p1["p_home_win"] + p2["p_away_win"]) / 2
    p_draw = (p1["p_draw"] + p2["p_draw"]) / 2
    p_opp_win = (p1["p_away_win"] + p2["p_home_win"]) / 2

    probs = np.array([p_team_win, p_draw, p_opp_win], dtype=float)
    probs = np.clip(probs, 1e-6, 1.0)
    probs = probs / probs.sum()

    grid, _ = calibrated_score_grid(
        team_xg,
        opp_xg,
        clf_probs=probs,
        max_goals=max_goals,
    )

    top = []
    for h in range(max_goals + 1):
        for a in range(max_goals + 1):
            top.append((h, a, float(grid[h, a])))

    top = sorted(top, key=lambda x: x[2], reverse=True)

    return {
        "team": team,
        "opponent": opponent,
        "team_xg_pred": float(team_xg),
        "opponent_xg_pred": float(opp_xg),
        "home_xg_pred": float(team_xg),
        "away_xg_pred": float(opp_xg),
        "p_team_win": float(probs[0]),
        "p_draw": float(probs[1]),
        "p_opponent_win": float(probs[2]),
        "p_home_win": float(probs[0]),
        "p_away_win": float(probs[2]),
        "most_likely_score": f"{top[0][0]}:{top[0][1]}",
        "most_likely_score_prob": top[0][2],
        "top_scorelines": top[:12],
        "score_grid": grid,
    }

# Alte v2-Namen bleiben optional auch verfügbar, falls irgendwo noch eine Zelle sie nutzt.
predict_match_v2 = predict_match
predict_neutral_v2 = predict_neutral

print("Prediction helpers ready:")
print("- raw_predict_home_away(...)")
print("- predict_home_away(...)")
print("- predict_match(...)")
print("- predict_neutral(...)")


## 12. Deep-v1-Klasse definieren


In [ ]:
# Zelle 13: Code
# Was diese Zelle macht:
# Definiert nur die PyTorch-Klasse, die zum Laden des gespeicherten Deep-v1-Modells benötigt wird. Kein Training.

import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU erkannt: Inference nutzt CUDA.")
else:
    print("Keine GPU erkannt: Inference laeuft auf CPU, nur Monte Carlo kann langsamer sein.")

class FootballDeepV1(nn.Module):
    def __init__(
        self,
        num_dim,
        n_teams,
        n_tournaments,
        n_confs,
        seq_dim,
        team_emb_dim=16,
        tourn_emb_dim=6,
        conf_emb_dim=4,
        seq_hidden=24,
        hidden=160,
        dropout=0.34,
    ):
        super().__init__()
        self.home_team_emb = nn.Embedding(n_teams, team_emb_dim)
        self.away_team_emb = nn.Embedding(n_teams, team_emb_dim)
        self.tournament_emb = nn.Embedding(n_tournaments, tourn_emb_dim)
        self.home_conf_emb = nn.Embedding(n_confs, conf_emb_dim)
        self.away_conf_emb = nn.Embedding(n_confs, conf_emb_dim)

        self.seq_gru = nn.GRU(
            input_size=seq_dim,
            hidden_size=seq_hidden,
            batch_first=True,
            bidirectional=True,
        )

        cat_dim = team_emb_dim * 2 + tourn_emb_dim + conf_emb_dim * 2
        seq_out_dim = seq_hidden * 4
        input_dim = num_dim + cat_dim + seq_out_dim

        self.trunk = nn.Sequential(
            nn.Linear(input_dim, hidden),
            nn.LayerNorm(hidden),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, hidden),
            nn.LayerNorm(hidden),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, hidden // 2),
            nn.SiLU(),
        )

        self.goal_head = nn.Linear(hidden // 2, 2)
        self.outcome_head = nn.Linear(hidden // 2, 3)
        self.draw_head = nn.Linear(hidden // 2, 1)
        self.total_head = nn.Linear(hidden // 2, 1)
        self.diff_head = nn.Linear(hidden // 2, 1)

    def encode_seq(self, seq):
        _, h = self.seq_gru(seq)
        return h.transpose(0, 1).reshape(seq.shape[0], -1)

    def forward(self, x_num, home_team, away_team, tournament, home_conf, away_conf, home_seq_x, away_seq_x):
        home_s = self.encode_seq(home_seq_x)
        away_s = self.encode_seq(away_seq_x)

        x = torch.cat([
            x_num,
            self.home_team_emb(home_team),
            self.away_team_emb(away_team),
            self.tournament_emb(tournament),
            self.home_conf_emb(home_conf),
            self.away_conf_emb(away_conf),
            home_s,
            away_s,
        ], dim=1)

        z = self.trunk(x)
        goals = F.softplus(self.goal_head(z)) + 1e-4
        outcome_logits = self.outcome_head(z)
        draw_logit = self.draw_head(z).squeeze(1)
        total = F.softplus(self.total_head(z).squeeze(1))
        diff = self.diff_head(z).squeeze(1)

        return {
            "goals": goals,
            "outcome_logits": outcome_logits,
            "draw_logit": draw_logit,
            "total": total,
            "diff": diff,
        }


## 14. Deep-v1-Modell laden und Helper definieren


In [ ]:
# Zelle 15: Code
# Was diese Zelle macht:
# Definiert fixture_row(...) als saubere Brücke von der v2-Snapshot-Logik zum Deep-Hybrid-Helper. Verhindert Namenschaos im Kernel.

# Bridge: fixture_row für Deep/Hybrid
# Diese Zelle gibt dem Deep-Helper einen stabilen Namen für die v2-Snapshot-Funktion.
needed = ["snapshot_features_v2", "state", "norm_team", "fifa_before"]
missing = [x for x in needed if x not in globals()]
if missing:
    raise RuntimeError(f"Diese Sachen fehlen noch: {missing}. Erst v2 Feature Builder laufen lassen.")
def fixture_row(home, away, match_date, tournament="FIFA World Cup", country="United States", neutral=True):
    return snapshot_features_v2(home, away, match_date, tournament=tournament, country=country, neutral=neutral, state=state)
print("fixture_row ready.")
print("Germany FIFA sanity:", fixture_row("Germany", "Curacao", date(2026, 6, 14)).get("home_fifa_rank"))


In [ ]:
# Zelle 16: Code
# Was diese Zelle macht:
# Lädt Deep v1 und definiert predict_match_deep, predict_neutral_deep sowie predict_match_hybrid/predict_neutral_hybrid. Die Hybrid-Funktionen akzeptieren max_goals und sind Monte-Carlo-kompatibel.

# WeltmeisterKI Deep v1 - Prediction Helper
# Läuft nach deiner v2 Prediction Helper Zelle.
# Definiert:
#   predict_match_deep
#   predict_neutral_deep
#   predict_match_hybrid
#   predict_neutral_hybrid
# Optional: USE_DEEP_HYBRID_AS_DEFAULT = True

import numpy as np
import torch
import torch.nn.functional as F

V2_PREDICT_MATCH = globals().get("predict_match")
V2_PREDICT_NEUTRAL = globals().get("predict_neutral")

if "fixture_row" not in globals():
    if "snapshot_features_v2" in globals() and "state" in globals():
        def fixture_row(home, away, match_date, tournament="FIFA World Cup", country="United States", neutral=True):
            return snapshot_features_v2(home, away, match_date, tournament=tournament, country=country, neutral=neutral, state=state)
    else:
        raise RuntimeError("Bitte zuerst v2 Feature Builder und v2 Prediction Helper ausführen, damit fixture_row/snapshot_features_v2 existiert.")

deep_bundle = torch.load(MODELS / "weltmeisterki_deep_v1.pt", map_location=device, weights_only=False)

deep_model = FootballDeepV1(**deep_bundle["model_config"]).to(device)
deep_model.load_state_dict(deep_bundle["state_dict"])
deep_model.eval()

deep_feature_cols = deep_bundle["feature_cols"]
team_to_id = deep_bundle["team_to_id"]
tournament_to_id = deep_bundle["tournament_to_id"]
conf_to_id = deep_bundle["conf_to_id"]
conf_map = deep_bundle["conf_map"]
SEQ_LEN = deep_bundle["seq_len"]
SEQ_COLS = deep_bundle["seq_cols"]
seq_mean = deep_bundle["seq_mean"]
seq_std = deep_bundle["seq_std"]
team_seq_history = deep_bundle["team_seq_history"]
num_imputer = deep_bundle["num_imputer"]
num_scaler = deep_bundle["num_scaler"]
deep_calibration = deep_bundle["calibration"]

def deep_conf_of(team):
    return conf_map.get(norm_team(team), "UNKNOWN")

def tournament_to_deep_id(tournament):
    tournament = str(tournament or "Other")
    if tournament in tournament_to_id:
        return tournament_to_id[tournament]
    if "World Cup" in tournament:
        return tournament_to_id.get("FIFA World Cup", 0)
    return tournament_to_id.get("Other", 0)

def deep_transform_one_seq(team):
    hist = team_seq_history.get(norm_team(team), [])
    arr = np.full((SEQ_LEN, len(SEQ_COLS)), np.nan, dtype=np.float32)
    xs = hist[-SEQ_LEN:]
    if xs:
        arr[-len(xs):, :] = np.array(xs, dtype=np.float32)
    z = (arr - seq_mean) / seq_std
    return np.nan_to_num(z, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

def deep_fixture_tensors(home, away, match_date, tournament="FIFA World Cup", country="United States", neutral=True):
    row = fixture_row(home, away, match_date, tournament=tournament, country=country, neutral=neutral)

    x_num = np.array([[row.get(c, np.nan) for c in deep_feature_cols]], dtype=np.float32)
    x_num = num_scaler.transform(num_imputer.transform(x_num)).astype(np.float32)

    ht = team_to_id.get(norm_team(home), 0)
    at = team_to_id.get(norm_team(away), 0)
    tid = tournament_to_deep_id(tournament)
    hc = conf_to_id.get(deep_conf_of(home), 0)
    ac = conf_to_id.get(deep_conf_of(away), 0)

    hsx = deep_transform_one_seq(home)[None, :, :]
    asx = deep_transform_one_seq(away)[None, :, :]

    return [
        torch.tensor(x_num, dtype=torch.float32).to(device),
        torch.tensor([ht], dtype=torch.long).to(device),
        torch.tensor([at], dtype=torch.long).to(device),
        torch.tensor([tid], dtype=torch.long).to(device),
        torch.tensor([hc], dtype=torch.long).to(device),
        torch.tensor([ac], dtype=torch.long).to(device),
        torch.tensor(hsx, dtype=torch.float32).to(device),
        torch.tensor(asx, dtype=torch.float32).to(device),
    ]

def calibrated_deep_score_matrix(home_lam, away_lam, clf_probs, draw_prob, max_goals=10):
    mat = raw_poisson_matrix(
        home_lam,
        away_lam,
        max_goals=max_goals,
        lambda_scale=deep_calibration["lambda_scale"],
        draw_boost=deep_calibration["draw_boost"],
    )

    pois = outcome_from_matrix(mat)
    d = float(draw_prob)
    side_sum = max(pois[0] + pois[2], 1e-6)

    draw_vec = np.array([
        (1.0 - d) * pois[0] / side_sum,
        d,
        (1.0 - d) * pois[2] / side_sum,
    ])

    target = (
        deep_calibration["poisson_weight"] * pois
        + deep_calibration["classifier_weight"] * clf_probs
        + deep_calibration["draw_weight"] * draw_vec
    )
    target = np.clip(target, 1e-6, 1.0)
    target = target / target.sum()

    return reconcile_score_matrix(mat, target)

def score_summary_from_grid(mat):
    p_home = float(np.tril(mat, -1).sum())
    p_draw = float(np.trace(mat))
    p_away = float(np.triu(mat, 1).sum())

    pairs = []
    for h in range(mat.shape[0]):
        for a in range(mat.shape[1]):
            pairs.append((h, a, float(mat[h, a])))

    pairs = sorted(pairs, key=lambda x: x[2], reverse=True)
    ml = pairs[0]

    return {
        "p_home_win": p_home,
        "p_draw": p_draw,
        "p_away_win": p_away,
        "most_likely_score": f"{ml[0]}:{ml[1]}",
        "most_likely_score_prob": ml[2],
        "top_scorelines": pairs[:12],
        "score_grid": mat,
    }

DEEP_UNCERTAINTY_SAMPLES = 12

@torch.no_grad()
def deep_raw_predict_home_away(home, away, match_date, tournament="FIFA World Cup", country="United States", neutral=True, max_goals=10, uncertainty_samples=None):
    if uncertainty_samples is None:
        uncertainty_samples = DEEP_UNCERTAINTY_SAMPLES
    xs = deep_fixture_tensors(home, away, match_date, tournament=tournament, country=country, neutral=neutral)

    mats = []
    home_lams = []
    away_lams = []

    # Dropout + LogNormal-Lambda-Jitter = Modellunsicherheit für Monte Carlo.
    deep_model.train() if uncertainty_samples > 1 else deep_model.eval()

    for _ in range(max(1, uncertainty_samples)):
        out = deep_model(*xs)
        goals = out["goals"].detach().cpu().numpy()[0]
        logits = out["outcome_logits"].detach().cpu().numpy()
        clf = softmax_np(logits)[0]
        draw_p = torch.sigmoid(out["draw_logit"]).detach().cpu().numpy()[0]

        jitter_h = np.random.lognormal(mean=0.0, sigma=0.10) if uncertainty_samples > 1 else 1.0
        jitter_a = np.random.lognormal(mean=0.0, sigma=0.10) if uncertainty_samples > 1 else 1.0

        hl = float(goals[0]) * jitter_h
        al = float(goals[1]) * jitter_a

        mats.append(calibrated_deep_score_matrix(hl, al, clf, draw_p, max_goals=max_goals))
        home_lams.append(hl)
        away_lams.append(al)

    deep_model.eval()

    mat = np.mean(mats, axis=0)
    mat = mat / mat.sum()

    out = score_summary_from_grid(mat)
    out["home_xg_pred"] = float(np.mean(home_lams))
    out["away_xg_pred"] = float(np.mean(away_lams))
    out["home_xg"] = out["home_xg_pred"]
    out["away_xg"] = out["away_xg_pred"]
    out["model"] = "deep_v1"
    return out

def predict_match_deep(home, away, match_date, tournament="FIFA World Cup", country="United States", neutral=True, max_goals=10, uncertainty_samples=None):
    return deep_raw_predict_home_away(home, away, match_date, tournament, country, neutral, max_goals=max_goals, uncertainty_samples=uncertainty_samples)

def predict_neutral_deep(team, opponent, match_date, tournament="FIFA World Cup", country="United States", max_goals=10, uncertainty_samples=None):
    p1 = deep_raw_predict_home_away(team, opponent, match_date, tournament, country, True, max_goals=max_goals, uncertainty_samples=uncertainty_samples)
    p2 = deep_raw_predict_home_away(opponent, team, match_date, tournament, country, True, max_goals=max_goals, uncertainty_samples=uncertainty_samples)

    mat = 0.5 * p1["score_grid"] + 0.5 * p2["score_grid"].T
    mat = mat / mat.sum()

    out = score_summary_from_grid(mat)
    out["team_xg_pred"] = 0.5 * (p1["home_xg_pred"] + p2["away_xg_pred"])
    out["opponent_xg_pred"] = 0.5 * (p1["away_xg_pred"] + p2["home_xg_pred"])
    out["p_team_win"] = out.pop("p_home_win")
    out["p_opponent_win"] = out.pop("p_away_win")
    out["model"] = "deep_v1_neutral"
    return out

def grid_from_prediction(pred):
    if "score_grid" in pred:
        mat = np.asarray(pred["score_grid"], dtype=np.float64)
        return mat / mat.sum()

    # Fallback, falls altes v2 kein score_grid enthält.
    h = pred.get("home_xg_pred", pred.get("team_xg_pred"))
    a = pred.get("away_xg_pred", pred.get("opponent_xg_pred"))
    return raw_poisson_matrix(h, a, max_goals=10, lambda_scale=1.0, draw_boost=1.15)

HYBRID_DEEP_WEIGHT = 0.30

def predict_match_hybrid(home, away, match_date, tournament="FIFA World Cup", country="United States", neutral=True, max_goals=10, uncertainty_samples=None):
    if V2_PREDICT_MATCH is None:
        raise RuntimeError("V2_PREDICT_MATCH fehlt. Bitte vorher deine v2 Prediction Helper Zelle laufen lassen.")

    if HYBRID_DEEP_WEIGHT <= 0:
        return V2_PREDICT_MATCH(home, away, match_date, tournament=tournament, country=country, neutral=neutral, max_goals=max_goals)
    if HYBRID_DEEP_WEIGHT >= 1:
        return predict_match_deep(home, away, match_date, tournament=tournament, country=country, neutral=neutral, max_goals=max_goals, uncertainty_samples=uncertainty_samples)

    v2 = V2_PREDICT_MATCH(home, away, match_date, tournament=tournament, country=country, neutral=neutral, max_goals=max_goals)
    dp = predict_match_deep(home, away, match_date, tournament=tournament, country=country, neutral=neutral, max_goals=max_goals, uncertainty_samples=uncertainty_samples)

    mat = (1.0 - HYBRID_DEEP_WEIGHT) * grid_from_prediction(v2) + HYBRID_DEEP_WEIGHT * grid_from_prediction(dp)
    mat = mat / mat.sum()

    out = score_summary_from_grid(mat)
    v2_home_xg = v2.get("home_xg_pred", v2.get("home_xg"))
    v2_away_xg = v2.get("away_xg_pred", v2.get("away_xg"))
    out["home_xg_pred"] = (1.0 - HYBRID_DEEP_WEIGHT) * v2_home_xg + HYBRID_DEEP_WEIGHT * dp["home_xg_pred"]
    out["away_xg_pred"] = (1.0 - HYBRID_DEEP_WEIGHT) * v2_away_xg + HYBRID_DEEP_WEIGHT * dp["away_xg_pred"]
    out["home_xg"] = out["home_xg_pred"]
    out["away_xg"] = out["away_xg_pred"]
    out["model"] = "hybrid_v2_plus_deep_v1"
    return out

def predict_neutral_hybrid(team, opponent, match_date, tournament="FIFA World Cup", country="United States", max_goals=10, uncertainty_samples=None):
    if V2_PREDICT_NEUTRAL is None:
        raise RuntimeError("V2_PREDICT_NEUTRAL fehlt. Bitte vorher deine v2 Prediction Helper Zelle laufen lassen.")

    if HYBRID_DEEP_WEIGHT <= 0:
        return V2_PREDICT_NEUTRAL(team, opponent, match_date, tournament=tournament, country=country, max_goals=max_goals)
    if HYBRID_DEEP_WEIGHT >= 1:
        return predict_neutral_deep(team, opponent, match_date, tournament=tournament, country=country, max_goals=max_goals, uncertainty_samples=uncertainty_samples)

    v2 = V2_PREDICT_NEUTRAL(team, opponent, match_date, tournament=tournament, country=country, max_goals=max_goals)
    dp = predict_neutral_deep(team, opponent, match_date, tournament=tournament, country=country, max_goals=max_goals, uncertainty_samples=uncertainty_samples)

    mat = (1.0 - HYBRID_DEEP_WEIGHT) * grid_from_prediction(v2) + HYBRID_DEEP_WEIGHT * grid_from_prediction(dp)
    mat = mat / mat.sum()

    out = score_summary_from_grid(mat)
    out["team_xg_pred"] = (1.0 - HYBRID_DEEP_WEIGHT) * v2["team_xg_pred"] + HYBRID_DEEP_WEIGHT * dp["team_xg_pred"]
    out["opponent_xg_pred"] = (1.0 - HYBRID_DEEP_WEIGHT) * v2["opponent_xg_pred"] + HYBRID_DEEP_WEIGHT * dp["opponent_xg_pred"]
    out["p_team_win"] = out.pop("p_home_win")
    out["p_opponent_win"] = out.pop("p_away_win")
    out["model"] = "hybrid_v2_plus_deep_v1_neutral"
    return out

USE_DEEP_HYBRID_AS_DEFAULT = False

if USE_DEEP_HYBRID_AS_DEFAULT:
    predict_match = predict_match_hybrid
    predict_neutral = predict_neutral_hybrid
    print("Default predictions now use HYBRID v2 + Deep v1.")
else:
    print("Deep ready. Nutze predict_neutral_deep(...) oder predict_neutral_hybrid(...).")


## 17. Finale Hybrid-Prognose aktivieren


In [ ]:
# Zelle 18: Code
# Was diese Zelle macht:
# Lädt das gespeicherte V4/Hybrid-Bundle und definiert Scoreline-Kalibrierungsfunktionen für das finale Modell.

import joblib
import numpy as np

v4_bundle_path = MODELS / "weltmeisterki4_sota_meta.joblib"
v4_bundle = joblib.load(v4_bundle_path)
v4_best_fixed_deep_weight = float(v4_bundle.get("best_fixed_deep_weight", 0.35))
v4_scoreline_calibration = v4_bundle.get("scoreline_calibration", {
    "low_score_boost": 1.05, "dc_00": 1.2, "dc_10": 1.1, "dc_01": 1.1, "dc_11": 1.05
})
V4_DEEP_UNCERTAINTY_SAMPLES = 8

def _as_grid(pred):
    g = np.asarray(pred["score_grid"], dtype=np.float64).copy()
    g = np.clip(g, 1e-12, None)
    return g / g.sum()

def outcome_from_matrix(mat):
    return np.array([
        float(np.tril(mat, -1).sum()),
        float(np.trace(mat)),
        float(np.triu(mat, 1).sum()),
    ], dtype=np.float64)

def reconcile_score_matrix(mat, target_probs):
    target = np.asarray(target_probs, dtype=np.float64)
    target = np.clip(target, 1e-6, 1.0)
    target = target / target.sum()
    masks = [
        np.tril(np.ones_like(mat), -1).astype(bool),
        np.eye(mat.shape[0], dtype=bool),
        np.triu(np.ones_like(mat), 1).astype(bool),
    ]
    out = mat.copy()
    current = [out[m].sum() for m in masks]
    for i, m in enumerate(masks):
        if current[i] > 1e-12:
            out[m] *= target[i] / current[i]
    out = np.clip(out, 1e-12, None)
    return out / out.sum()

def apply_scoreline_adjustments(mat, cal):
    out = np.asarray(mat, dtype=np.float64).copy()
    max_g = out.shape[0] - 1
    low_boost = cal.get("low_score_boost", 1.0)
    for h in range(max_g + 1):
        for a in range(max_g + 1):
            if h + a <= 2:
                out[h, a] *= low_boost
    out[0, 0] *= cal.get("dc_00", 1.0)
    if max_g >= 1:
        out[1, 0] *= cal.get("dc_10", 1.0)
        out[0, 1] *= cal.get("dc_01", 1.0)
        out[1, 1] *= cal.get("dc_11", 1.0)
    out = np.clip(out, 1e-12, None)
    return out / out.sum()

def _summary_from_final_grid(mat):
    mat = np.asarray(mat, dtype=np.float64)
    mat = mat / mat.sum()
    p_home = float(np.tril(mat, -1).sum())
    p_draw = float(np.trace(mat))
    p_away = float(np.triu(mat, 1).sum())
    pairs = []
    for h in range(mat.shape[0]):
        for a in range(mat.shape[1]):
            pairs.append((h, a, float(mat[h, a])))
    pairs.sort(key=lambda x: x[2], reverse=True)
    return {
        "p_home_win": p_home, "p_draw": p_draw, "p_away_win": p_away,
        "most_likely_score": f"{pairs[0][0]}:{pairs[0][1]}",
        "most_likely_score_prob": pairs[0][2],
        "top_scorelines": pairs[:12],
        "score_grid": mat,
    }

print("Loaded V4 bundle:", v4_bundle_path)
print("Deep weight:", v4_best_fixed_deep_weight)
print("Scoreline calibration:", v4_scoreline_calibration)
if "performance_table" in v4_bundle:
    print("Performance table from training bundle:")
    print(v4_bundle["performance_table"])


In [ ]:
# Zelle 19: Code
# FINAL PRODUCTION MODEL: fixed_hybrid_35pct_deep + scoreline calibration
# Diese Zelle setzt das beste Modell nach LogLoss/Brier als aktives Modell.
# Danach nutzen Deutschland, Südkorea und Monte Carlo automatisch dieses Modell.

from datetime import date
import numpy as np

FIXED_HYBRID_DEEP_WEIGHT = float(v4_best_fixed_deep_weight)  # bei dir: 0.35

def predict_match_fixed_hybrid_scoreline(
    home,
    away,
    match_date,
    tournament="FIFA World Cup",
    country="United States",
    neutral=True,
    max_goals=10,
):
    v2 = predict_match_v2(
        home,
        away,
        match_date,
        tournament=tournament,
        country=country,
        neutral=neutral,
        max_goals=max_goals,
    )

    deep = predict_match_deep(
        home,
        away,
        match_date,
        tournament=tournament,
        country=country,
        neutral=neutral,
        max_goals=max_goals,
        uncertainty_samples=V4_DEEP_UNCERTAINTY_SAMPLES,
    )

    base_grid = (
        (1.0 - FIXED_HYBRID_DEEP_WEIGHT) * _as_grid(v2)
        + FIXED_HYBRID_DEEP_WEIGHT * _as_grid(deep)
    )
    base_grid = base_grid / base_grid.sum()

    target_probs = outcome_from_matrix(base_grid)

    adjusted_grid = apply_scoreline_adjustments(base_grid, v4_scoreline_calibration)
    final_grid = reconcile_score_matrix(adjusted_grid, target_probs)

    out = _summary_from_final_grid(final_grid)

    v2_home_xg = v2.get("home_xg_pred", v2.get("home_xg"))
    v2_away_xg = v2.get("away_xg_pred", v2.get("away_xg"))

    out["home"] = home
    out["away"] = away
    out["home_xg_pred"] = (
        (1.0 - FIXED_HYBRID_DEEP_WEIGHT) * v2_home_xg
        + FIXED_HYBRID_DEEP_WEIGHT * deep["home_xg_pred"]
    )
    out["away_xg_pred"] = (
        (1.0 - FIXED_HYBRID_DEEP_WEIGHT) * v2_away_xg
        + FIXED_HYBRID_DEEP_WEIGHT * deep["away_xg_pred"]
    )
    out["home_xg"] = out["home_xg_pred"]
    out["away_xg"] = out["away_xg_pred"]
    out["model"] = "fixed_hybrid_35pct_deep_scoreline"

    return out


def predict_neutral_fixed_hybrid_scoreline(
    team,
    opponent,
    match_date,
    tournament="FIFA World Cup",
    country="United States",
    max_goals=10,
):
    p1 = predict_match_fixed_hybrid_scoreline(
        team,
        opponent,
        match_date,
        tournament=tournament,
        country=country,
        neutral=True,
        max_goals=max_goals,
    )

    p2 = predict_match_fixed_hybrid_scoreline(
        opponent,
        team,
        match_date,
        tournament=tournament,
        country=country,
        neutral=True,
        max_goals=max_goals,
    )

    mat = 0.5 * p1["score_grid"] + 0.5 * p2["score_grid"].T
    mat = mat / mat.sum()

    out = _summary_from_final_grid(mat)

    out["team"] = team
    out["opponent"] = opponent
    out["team_xg_pred"] = 0.5 * (p1["home_xg_pred"] + p2["away_xg_pred"])
    out["opponent_xg_pred"] = 0.5 * (p1["away_xg_pred"] + p2["home_xg_pred"])
    out["home_xg_pred"] = out["team_xg_pred"]
    out["away_xg_pred"] = out["opponent_xg_pred"]

    out["p_team_win"] = out.pop("p_home_win")
    out["p_opponent_win"] = out.pop("p_away_win")
    out["p_home_win"] = out["p_team_win"]
    out["p_away_win"] = out["p_opponent_win"]

    out["model"] = "fixed_hybrid_35pct_deep_scoreline_neutral"

    return out


predict_match = predict_match_fixed_hybrid_scoreline
predict_neutral = predict_neutral_fixed_hybrid_scoreline

ACTIVE_MODEL_NAME = f"fixed_hybrid_{int(FIXED_HYBRID_DEEP_WEIGHT * 100)}pct_deep_scoreline"

if "dist_cache" in globals():
    dist_cache.clear()

print("=" * 80)
print("AKTIVES MODELL FÜR ALLE FOLGENDEN ZELLEN")
print("=" * 80)
print("ACTIVE_MODEL_NAME:", ACTIVE_MODEL_NAME)
print("predict_match:", predict_match.__name__)
print("predict_neutral:", predict_neutral.__name__)
print("Deep weight:", FIXED_HYBRID_DEEP_WEIGHT)
print("v2 weight:", 1.0 - FIXED_HYBRID_DEEP_WEIGHT)
print("Scoreline calibration:", v4_scoreline_calibration)

print("\nPerformance-Vergleich:")
print(v4_performance_table)

# Sanity Check an einem Spiel
sanity = predict_neutral(
    "Germany",
    "Curacao",
    date(2026, 6, 14),
    tournament="FIFA World Cup",
    country="United States",
)

print("\nSanity Check: Germany vs Curacao")
print("model:", sanity["model"])
print("Germany xG:", round(sanity["team_xg_pred"], 3))
print("Curacao xG:", round(sanity["opponent_xg_pred"], 3))
print("Germany win %:", round(100 * sanity["p_team_win"], 1))
print("Draw %:", round(100 * sanity["p_draw"], 1))
print("Curacao win %:", round(100 * sanity["p_opponent_win"], 1))
print("Most likely score:", sanity["most_likely_score"])

print("\nJetzt kannst du Deutschland-Vorrunde, Südkorea und Monte Carlo neu starten.")


## 20. Bereits gespielte WM-Spiele eintragen


In [ ]:
# Zelle 21: Code
# Was diese Zelle macht:
# Hier werden alle 72 Gruppenspiele gepflegt. Bereits bekannte Ergebnisse sind eingetragen; bei offenen Spielen bleiben die Scores auf None.
# Wenn neue Ergebnisse dazukommen, nur home_score/away_score ersetzen, diese Zelle ausf?hren und danach die Prognosezellen erneut laufen lassen.

from collections import deque
from datetime import date
import unicodedata
import re
import numpy as np
import polars as pl

REAL_MATCH_STATE_REPEATS = 2

ALL_GROUP_MATCHES_INPUT = [
    {"match_number": 1,  "group": "A", "date": "2026-06-11", "time": None,    "home": "Mexico",       "away": "South Africa",          "home_score": 2,    "away_score": 0},
    {"match_number": 2,  "group": "A", "date": "2026-06-12", "time": None,    "home": "South Korea",  "away": "Czech Republic",         "home_score": 2,    "away_score": 1},
    {"match_number": 3,  "group": "B", "date": "2026-06-12", "time": None,    "home": "Canada",       "away": "Bosnia and Herzegovina", "home_score": 1,    "away_score": 1},
    {"match_number": 4,  "group": "D", "date": "2026-06-13", "time": None,    "home": "USA",          "away": "Paraguay",               "home_score": 4,    "away_score": 1},
    {"match_number": 5,  "group": "B", "date": "2026-06-13", "time": None,    "home": "Qatar",        "away": "Switzerland",            "home_score": 1,    "away_score": 1},
    {"match_number": 6,  "group": "C", "date": "2026-06-14", "time": None,    "home": "Brazil",       "away": "Morocco",                "home_score": 1,    "away_score": 1},
    {"match_number": 7,  "group": "C", "date": "2026-06-14", "time": None,    "home": "Haiti",        "away": "Scotland",               "home_score": 0,    "away_score": 1},
    {"match_number": 8,  "group": "D", "date": "2026-06-14", "time": None,    "home": "Australia",    "away": "Turkey",                 "home_score": 2,    "away_score": 0},
    {"match_number": 9,  "group": "E", "date": "2026-06-14", "time": None,    "home": "Germany",      "away": "Curacao",                "home_score": 7,    "away_score": 1},
    {"match_number": 10, "group": "F", "date": "2026-06-14", "time": None,    "home": "Netherlands",  "away": "Japan",                  "home_score": 2,    "away_score": 2},
    {"match_number": 11, "group": "E", "date": "2026-06-15", "time": None,    "home": "Ivory Coast",  "away": "Ecuador",                "home_score": 1,    "away_score": 0},
    {"match_number": 12, "group": "F", "date": "2026-06-15", "time": None,    "home": "Sweden",       "away": "Tunisia",                "home_score": 5,    "away_score": 1},
    {"match_number": 13, "group": "H", "date": "2026-06-15", "time": None,    "home": "Spain",        "away": "Cape Verde",             "home_score": 0,    "away_score": 0},
    {"match_number": 14, "group": "G", "date": "2026-06-15", "time": None,    "home": "Belgium",      "away": "Egypt",                  "home_score": 1,    "away_score": 1},
    {"match_number": 15, "group": "H", "date": "2026-06-16", "time": None,    "home": "Saudi Arabia", "away": "Uruguay",                "home_score": 1,    "away_score": 1},
    {"match_number": 16, "group": "G", "date": "2026-06-16", "time": None,    "home": "IR Iran",      "away": "New Zealand",            "home_score": 2,    "away_score": 2},
    {"match_number": 17, "group": "I", "date": "2026-06-16", "time": None,    "home": "France",       "away": "Senegal",                "home_score": 3,    "away_score": 1},
    {"match_number": 18, "group": "I", "date": "2026-06-17", "time": None,    "home": "Iraq",         "away": "Norway",                 "home_score": 1,    "away_score": 4},
    {"match_number": 19, "group": "J", "date": "2026-06-17", "time": None,    "home": "Argentina",    "away": "Algeria",                "home_score": 3,    "away_score": 0},
    {"match_number": 20, "group": "J", "date": "2026-06-17", "time": None,    "home": "Austria",      "away": "Jordan",                 "home_score": 3,    "away_score": 1},
    {"match_number": 21, "group": "K", "date": "2026-06-17", "time": None,    "home": "Portugal",     "away": "DR Congo",               "home_score": 1,    "away_score": 1},
    {"match_number": 22, "group": "L", "date": "2026-06-17", "time": None,    "home": "England",      "away": "Croatia",                "home_score": 4,    "away_score": 2},
    {"match_number": 23, "group": "L", "date": "2026-06-18", "time": None,    "home": "Ghana",        "away": "Panama",                 "home_score": 1,    "away_score": 0},
    {"match_number": 24, "group": "K", "date": "2026-06-18", "time": None,    "home": "Uzbekistan",   "away": "Colombia",               "home_score": 1,    "away_score": 3},
    {"match_number": 25, "group": "A", "date": "2026-06-18", "time": None,    "home": "Czech Republic", "away": "South Africa",        "home_score": 1,    "away_score": 1},
    {"match_number": 26, "group": "B", "date": "2026-06-18", "time": None,    "home": "Switzerland",  "away": "Bosnia and Herzegovina", "home_score": 4,    "away_score": 1},
    {"match_number": 27, "group": "B", "date": "2026-06-19", "time": None,    "home": "Canada",       "away": "Qatar",                  "home_score": 6,    "away_score": 0},
    {"match_number": 28, "group": "A", "date": "2026-06-19", "time": None,    "home": "Mexico",       "away": "South Korea",            "home_score": 1,    "away_score": 0},
    {"match_number": 29, "group": "D", "date": "2026-06-19", "time": None,    "home": "USA",          "away": "Australia",              "home_score": 2,    "away_score": 0},
    {"match_number": 30, "group": "C", "date": "2026-06-20", "time": None,    "home": "Scotland",     "away": "Morocco",                "home_score": 0,    "away_score": 1},
    {"match_number": 31, "group": "C", "date": "2026-06-20", "time": None,    "home": "Brazil",       "away": "Haiti",                  "home_score": 3,    "away_score": 0},
    {"match_number": 32, "group": "D", "date": "2026-06-20", "time": None,    "home": "Turkey",       "away": "Paraguay",               "home_score": 0,    "away_score": 1},
    {"match_number": 33, "group": "F", "date": "2026-06-20", "time": None,    "home": "Netherlands",  "away": "Sweden",                 "home_score": 5,    "away_score": 1},
    {"match_number": 34, "group": "E", "date": "2026-06-20", "time": None,    "home": "Germany",      "away": "Ivory Coast",            "home_score": 2,    "away_score": 1},
    {"match_number": 35, "group": "E", "date": "2026-06-21", "time": None,    "home": "Ecuador",      "away": "Curacao",                "home_score": 0,    "away_score": 0},
    {"match_number": 36, "group": "F", "date": "2026-06-21", "time": None,    "home": "Tunisia",      "away": "Japan",                  "home_score": 0,    "away_score": 4},
    {"match_number": 37, "group": "H", "date": "2026-06-21", "time": "18:00", "home": "Spain",        "away": "Saudi Arabia",          "home_score": None, "away_score": None},
    {"match_number": 38, "group": "G", "date": "2026-06-21", "time": "21:00", "home": "Belgium",      "away": "IR Iran",               "home_score": None, "away_score": None},
    {"match_number": 39, "group": "H", "date": "2026-06-22", "time": "00:00", "home": "Uruguay",      "away": "Cape Verde",            "home_score": None, "away_score": None},
    {"match_number": 40, "group": "G", "date": "2026-06-22", "time": "03:00", "home": "New Zealand",  "away": "Egypt",                 "home_score": None, "away_score": None},
    {"match_number": 41, "group": "J", "date": "2026-06-22", "time": "19:00", "home": "Argentina",    "away": "Austria",               "home_score": None, "away_score": None},
    {"match_number": 42, "group": "I", "date": "2026-06-22", "time": "23:00", "home": "France",       "away": "Iraq",                  "home_score": None, "away_score": None},
    {"match_number": 43, "group": "I", "date": "2026-06-23", "time": "02:00", "home": "Norway",       "away": "Senegal",               "home_score": None, "away_score": None},
    {"match_number": 44, "group": "J", "date": "2026-06-23", "time": "05:00", "home": "Jordan",       "away": "Algeria",               "home_score": None, "away_score": None},
    {"match_number": 45, "group": "K", "date": "2026-06-23", "time": "19:00", "home": "Portugal",     "away": "Uzbekistan",            "home_score": None, "away_score": None},
    {"match_number": 46, "group": "L", "date": "2026-06-23", "time": "22:00", "home": "England",      "away": "Ghana",                 "home_score": None, "away_score": None},
    {"match_number": 47, "group": "L", "date": "2026-06-24", "time": "01:00", "home": "Panama",       "away": "Croatia",               "home_score": None, "away_score": None},
    {"match_number": 48, "group": "K", "date": "2026-06-24", "time": "04:00", "home": "Colombia",     "away": "DR Congo",              "home_score": None, "away_score": None},
    {"match_number": 49, "group": "B", "date": "2026-06-24", "time": "21:00", "home": "Switzerland",  "away": "Canada",                "home_score": None, "away_score": None},
    {"match_number": 50, "group": "B", "date": "2026-06-24", "time": "21:00", "home": "Bosnia and Herzegovina", "away": "Qatar",       "home_score": None, "away_score": None},
    {"match_number": 51, "group": "C", "date": "2026-06-25", "time": "00:00", "home": "Scotland",     "away": "Brazil",                "home_score": None, "away_score": None},
    {"match_number": 52, "group": "C", "date": "2026-06-25", "time": "00:00", "home": "Morocco",      "away": "Haiti",                 "home_score": None, "away_score": None},
    {"match_number": 53, "group": "A", "date": "2026-06-25", "time": "03:00", "home": "Czech Republic", "away": "Mexico",            "home_score": None, "away_score": None},
    {"match_number": 54, "group": "A", "date": "2026-06-25", "time": "03:00", "home": "South Africa", "away": "South Korea",          "home_score": None, "away_score": None},
    {"match_number": 55, "group": "E", "date": "2026-06-25", "time": "22:00", "home": "Curacao",      "away": "Ivory Coast",           "home_score": None, "away_score": None},
    {"match_number": 56, "group": "E", "date": "2026-06-25", "time": "22:00", "home": "Ecuador",      "away": "Germany",               "home_score": None, "away_score": None},
    {"match_number": 57, "group": "F", "date": "2026-06-26", "time": "01:00", "home": "Japan",        "away": "Sweden",                "home_score": None, "away_score": None},
    {"match_number": 58, "group": "F", "date": "2026-06-26", "time": "01:00", "home": "Tunisia",      "away": "Netherlands",           "home_score": None, "away_score": None},
    {"match_number": 59, "group": "D", "date": "2026-06-26", "time": "04:00", "home": "Turkey",       "away": "USA",                   "home_score": None, "away_score": None},
    {"match_number": 60, "group": "D", "date": "2026-06-26", "time": "04:00", "home": "Paraguay",     "away": "Australia",             "home_score": None, "away_score": None},
    {"match_number": 61, "group": "I", "date": "2026-06-26", "time": "21:00", "home": "Norway",       "away": "France",                "home_score": None, "away_score": None},
    {"match_number": 62, "group": "I", "date": "2026-06-26", "time": "21:00", "home": "Senegal",      "away": "Iraq",                  "home_score": None, "away_score": None},
    {"match_number": 63, "group": "H", "date": "2026-06-27", "time": "02:00", "home": "Cape Verde",   "away": "Saudi Arabia",          "home_score": None, "away_score": None},
    {"match_number": 64, "group": "H", "date": "2026-06-27", "time": "02:00", "home": "Uruguay",      "away": "Spain",                 "home_score": None, "away_score": None},
    {"match_number": 65, "group": "G", "date": "2026-06-27", "time": "05:00", "home": "Egypt",        "away": "IR Iran",               "home_score": None, "away_score": None},
    {"match_number": 66, "group": "G", "date": "2026-06-27", "time": "05:00", "home": "New Zealand",  "away": "Belgium",               "home_score": None, "away_score": None},
    {"match_number": 67, "group": "L", "date": "2026-06-27", "time": "23:00", "home": "Panama",       "away": "England",               "home_score": None, "away_score": None},
    {"match_number": 68, "group": "L", "date": "2026-06-27", "time": "23:00", "home": "Croatia",      "away": "Ghana",                 "home_score": None, "away_score": None},
    {"match_number": 69, "group": "J", "date": "2026-06-28", "time": None,    "home": "Algeria",      "away": "Austria",               "home_score": None, "away_score": None},
    {"match_number": 70, "group": "J", "date": "2026-06-28", "time": None,    "home": "Jordan",       "away": "Argentina",             "home_score": None, "away_score": None},
    {"match_number": 71, "group": "K", "date": "2026-06-28", "time": None,    "home": "Colombia",     "away": "Portugal",              "home_score": None, "away_score": None},
    {"match_number": 72, "group": "K", "date": "2026-06-28", "time": None,    "home": "DR Congo",     "away": "Uzbekistan",            "home_score": None, "away_score": None},
]

TEAM_ALIASES_LIVE_EXACT = {
    "Cura?ao": "Curacao",
    "Curacao": "Curacao",
    "C?te d'Ivoire": "Ivory Coast",
    "C?te d?Ivoire": "Ivory Coast",
    "Cote d'Ivoire": "Ivory Coast",
    "Elfenbeink?ste": "Ivory Coast",
    "Ivory Coast": "Ivory Coast",
    "Republic of Korea": "South Korea",
    "Korea Republic": "South Korea",
    "Republik Korea": "South Korea",
    "South Korea": "South Korea",
    "Bosnien und Herzegowina": "Bosnia and Herzegovina",
    "Bosnia-Herzegovina": "Bosnia and Herzegovina",
    "Bosnia and Herzegovina": "Bosnia and Herzegovina",
    "Tschechien": "Czech Republic",
    "Czechia": "Czech Republic",
    "Czech Republic": "Czech Republic",
    "United States": "USA",
    "USA": "USA",
    "US": "USA",
    "IR Iran": "IR Iran",
    "Iran": "IR Iran",
    "DR Congo": "DR Congo",
    "DR Kongo": "DR Congo",
    "Congo DR": "DR Congo",
    "Democratic Republic of the Congo": "DR Congo",
    "Cabo Verde": "Cape Verde",
    "Cape Verde": "Cape Verde",
    "Kap Verde": "Cape Verde",
    "Schweiz": "Switzerland",
    "Niederlande": "Netherlands",
    "T?rkei": "Turkey",
    "Schweden": "Sweden",
    "Tunesien": "Tunisia",
    "?gypten": "Egypt",
    "Saudi-Arabien": "Saudi Arabia",
    "Neuseeland": "New Zealand",
    "Frankreich": "France",
    "Norwegen": "Norway",
    "Argentinien": "Argentina",
    "?sterreich": "Austria",
    "Jordanien": "Jordan",
    "Irak": "Iraq",
    "Kanada": "Canada",
    "Mexiko": "Mexico",
    "S?dafrika": "South Africa",
    "Brasilien": "Brazil",
    "Marokko": "Morocco",
    "Schottland": "Scotland",
    "Belgien": "Belgium",
}

def _plain_team_key(x):
    x = "" if x is None else str(x).strip()
    x = unicodedata.normalize("NFKD", x).encode("ascii", "ignore").decode("ascii")
    return re.sub(r"[^a-zA-Z0-9]+", " ", x).strip().lower()

TEAM_ALIASES_LIVE_PLAIN = {_plain_team_key(k): v for k, v in TEAM_ALIASES_LIVE_EXACT.items()}

def canonical_team_live(team):
    raw = "" if team is None else str(team).strip()
    if raw in TEAM_ALIASES_LIVE_EXACT:
        return TEAM_ALIASES_LIVE_EXACT[raw]
    return TEAM_ALIASES_LIVE_PLAIN.get(_plain_team_key(raw), raw)

if "norm_team" in globals():
    if "ORIGINAL_NORM_TEAM_BEFORE_LIVE_ALIASES" not in globals():
        ORIGINAL_NORM_TEAM_BEFORE_LIVE_ALIASES = norm_team

    def norm_team(s):
        base = ORIGINAL_NORM_TEAM_BEFORE_LIVE_ALIASES(s)
        return canonical_team_live(base)

def _score_known(row):
    return row.get("home_score") is not None and row.get("away_score") is not None

def _parse_live_date(row):
    value = row.get("date")
    if isinstance(value, date):
        return value
    try:
        y, m, d = map(int, str(value).split("-"))
        return date(y, m, d)
    except Exception:
        return date(2026, 6, 1)

def normalize_group_matches(rows):
    out, seen = [], set()
    for r in rows:
        rr = dict(r)
        rr["match_number"] = int(rr["match_number"])
        rr["group"] = str(rr.get("group") or "")
        rr["home"] = canonical_team_live(rr["home"])
        rr["away"] = canonical_team_live(rr["away"])
        rr["home_score"] = None if rr.get("home_score") is None else int(rr["home_score"])
        rr["away_score"] = None if rr.get("away_score") is None else int(rr["away_score"])
        if rr["match_number"] in seen:
            raise ValueError(f"match_number doppelt: {rr['match_number']}")
        seen.add(rr["match_number"])
        out.append(rr)
    if len(out) != 72:
        raise ValueError(f"Es muessen 72 Gruppenspiele eingetragen sein, gefunden: {len(out)}")
    return sorted(out, key=lambda x: x["match_number"])

ALL_GROUP_MATCHES = normalize_group_matches(ALL_GROUP_MATCHES_INPUT)
PLAYED_MATCHES = [r for r in ALL_GROUP_MATCHES if _score_known(r)]
OPEN_GROUP_MATCHES = [r for r in ALL_GROUP_MATCHES if not _score_known(r)]
PLAYED_MATCHES_BY_NUMBER = {r["match_number"]: r for r in PLAYED_MATCHES}
PLAYED_MATCHES_VERSION = tuple(
    sorted((r["match_number"], r["home"], r["away"], r["home_score"], r["away_score"]) for r in PLAYED_MATCHES)
)

def get_played_match_result(match_number, home=None, away=None):
    return PLAYED_MATCHES_BY_NUMBER.get(int(match_number))

def _orient_played_result(played, home, away):
    ph = canonical_team_live(played["home"])
    pa = canonical_team_live(played["away"])
    ch = canonical_team_live(home)
    ca = canonical_team_live(away)
    hs = int(played["home_score"])
    aw = int(played["away_score"])
    if ph == ch and pa == ca:
        return hs, aw
    if ph == ca and pa == ch:
        return aw, hs
    raise ValueError(
        "Played result passt nicht zum CSV-Match. "
        f"played={ph} vs {pa}, csv={ch} vs {ca}. "
        "Falls das dasselbe Team ist: Alias oben in TEAM_ALIASES_LIVE_EXACT ergaenzen."
    )

def played_result_as_res(played, home, away, match_date, country, knockout=False, distribution_func=None):
    hs, aw = _orient_played_result(played, home, away)
    model_home = canonical_team_live(home)
    model_away = canonical_team_live(away)
    dist = None
    if distribution_func is not None:
        try:
            dist = distribution_func(model_home, model_away, match_date, country)
        except Exception as e:
            print("Warnung: Modellverteilung fuer gespieltes Match nicht berechnet:", played["match_number"], repr(e))
    if hs > aw:
        winner, loser = home, away
    elif aw > hs:
        winner, loser = away, home
    else:
        winner, loser = None, None
    if dist is None:
        p_home = p_draw = p_away = np.nan
        p_home_adv = p_away_adv = np.nan
        home_xg = away_xg = np.nan
        grid = None
        score_prob = np.nan
        most_likely_score = None
    else:
        p_home = float(dist["p_home_win"])
        p_draw = float(dist["p_draw"])
        p_away = float(dist["p_away_win"])
        home_xg = float(dist["home_xg"])
        away_xg = float(dist["away_xg"])
        grid = dist.get("grid")
        score_prob = float(grid[hs, aw]) if grid is not None and hs < grid.shape[0] and aw < grid.shape[1] else np.nan
        most_likely_score = dist.get("most_likely_score")
        denom = max(p_home + p_away, 1e-9)
        p_home_adv = p_home + p_draw * (p_home / denom)
        p_away_adv = p_away + p_draw * (p_away / denom)
    return {
        "home": home,
        "away": away,
        "home_score": int(hs),
        "away_score": int(aw),
        "score": f"{hs}:{aw}",
        "winner": winner,
        "loser": loser,
        "home_xg": home_xg,
        "away_xg": away_xg,
        "p_home_win": p_home,
        "p_draw": p_draw,
        "p_away_win": p_away,
        "p_home_advance": p_home_adv,
        "p_away_advance": p_away_adv,
        "score_grid": grid,
        "most_likely_score": most_likely_score,
        "most_likely_score_prob": score_prob,
        "score_prob_%": round(100 * score_prob, 2) if np.isfinite(score_prob) else None,
        "status": "Gespielt",
    }

def clone_state_v2(src):
    dst = init_state_v2()
    for key in ["elo", "attack", "def_weak", "shoot_w", "shoot_l"]:
        dst[key].update(dict(src[key]))
    for key in ["team_hist", "xg_hist", "scorer_hist", "h2h"]:
        for k, v in src[key].items():
            dst[key][k] = deque(list(v), maxlen=v.maxlen)
    return dst

def apply_played_matches_to_forecast_state(base_state=None, repeats=REAL_MATCH_STATE_REPEATS):
    global state, FORECAST_STATE, BASE_FORECAST_STATE, LIVE_ORIGINAL_MODEL_STATE
    if "LIVE_ORIGINAL_MODEL_STATE" not in globals():
        LIVE_ORIGINAL_MODEL_STATE = clone_state_v2(state)
    if base_state is None:
        base_state = LIVE_ORIGINAL_MODEL_STATE
    BASE_FORECAST_STATE = clone_state_v2(base_state)
    FORECAST_STATE = clone_state_v2(base_state)
    for r in sorted(PLAYED_MATCHES, key=lambda x: x["match_number"]):
        update_row = {
            "date": _parse_live_date(r),
            "home_team": canonical_team_live(r["home"]),
            "away_team": canonical_team_live(r["away"]),
            "home_score": int(r["home_score"]),
            "away_score": int(r["away_score"]),
            "tournament": "FIFA World Cup",
        }
        for _ in range(int(repeats)):
            update_state_after_match_v2(update_row, FORECAST_STATE)
    state = FORECAST_STATE
    for cache_name in ["dist_cache", "single_dist_cache"]:
        if cache_name in globals() and hasattr(globals()[cache_name], "clear"):
            globals()[cache_name].clear()

apply_played_matches_to_forecast_state()

all_group_matches_table = pl.DataFrame([
    {
        "match_number": r["match_number"],
        "group": r["group"],
        "date": r.get("date"),
        "time": r.get("time"),
        "home": r["home"],
        "away": r["away"],
        "home_score": r["home_score"],
        "away_score": r["away_score"],
        "status": "Gespielt" if _score_known(r) else "Offen",
    }
    for r in ALL_GROUP_MATCHES
])

played_matches_table = all_group_matches_table.filter(pl.col("status") == "Gespielt")
open_matches_table = all_group_matches_table.filter(pl.col("status") == "Offen")

print("Gruppenspiele insgesamt:", len(ALL_GROUP_MATCHES))
print("Bereits gespielte Matches:", len(PLAYED_MATCHES))
print("Offene Matches:", len(OPEN_GROUP_MATCHES))
print("REAL_MATCH_STATE_REPEATS:", REAL_MATCH_STATE_REPEATS)
print("Forecast-State wurde aktualisiert.")
print("Alias sanity:")
print("  Cape Verde ->", canonical_team_live("Cape Verde"))
print("  Cabo Verde ->", canonical_team_live("Cabo Verde"))
print("  Kap Verde ->", canonical_team_live("Kap Verde"))
print("  Cura?ao ->", canonical_team_live("Cura?ao"))
print("  Republic of Korea ->", canonical_team_live("Republic of Korea"))
print("\nAlle 72 Gruppenspiele:")
print(all_group_matches_table)
print("\nNoch offene Gruppenspiele:")
print(open_matches_table)


## 22. Deutschland Gruppenspiele


In [ ]:
# Zelle 23: Code
# Was diese Zelle macht:
# Zeigt Deutschlands Gruppenspiele mit Status: gespielte Spiele als echtes Ergebnis, offene Spiele als Modellprognose.
# Enthält Sieg/Remis/Niederlage, xG, wahrscheinlichsten Score und Top-8-Scorelines.

pl.Config.set_tbl_cols(30)
pl.Config.set_tbl_rows(80)
pl.Config.set_fmt_str_lengths(120)

fixtures = [
    {"match_number": 9,  "team": "Germany", "opponent": "Curacao",     "display": "Deutschland vs Curacao",        "date": date(2026, 6, 14), "country": "United States"},
    {"match_number": 34, "team": "Germany", "opponent": "Ivory Coast", "display": "Deutschland vs Elfenbeinküste", "date": date(2026, 6, 20), "country": "United States"},
    {"match_number": 56, "team": "Germany", "opponent": "Ecuador",     "display": "Deutschland vs Ecuador",       "date": date(2026, 6, 25), "country": "United States"},
]

def top8_from_prediction(pred):
    return ", ".join([f"{h}:{a} ({100*p:.1f}%)" for h, a, p in pred.get("top_scorelines", [])[:8]])

preds, score_rows = [], []
for fx in fixtures:
    fixed = get_played_match_result(fx["match_number"], fx["team"], fx["opponent"]) if "get_played_match_result" in globals() else None
    pred = predict_neutral(fx["team"], fx["opponent"], fx["date"], fx["country"])
    gw, dr, ow = pred["p_team_win"], pred["p_draw"], pred["p_opponent_win"]
    if fixed is not None:
        hs, aw = _orient_played_result(fixed, fx["team"], fx["opponent"])
        status = "Gespielt"
        result_score = f"{hs}:{aw}"
        pick = "Deutschland gewinnt" if hs > aw else "Unentschieden" if hs == aw else "Deutschland verliert"
    else:
        status = "Prognose"
        result_score = pred["most_likely_score"]
        pick = "Deutschland gewinnt" if gw > max(dr, ow) else "Unentschieden" if dr > max(gw, ow) else "Gegner gewinnt"
    preds.append({
        "match": fx["display"], "status": status, "score": result_score, "date": fx["date"],
        "deutschland_xg": round(pred["team_xg_pred"], 3), "gegner_xg": round(pred["opponent_xg_pred"], 3),
        "deutschland_sieg_%": round(100 * gw, 1), "remis_%": round(100 * dr, 1), "deutschland_niederlage_%": round(100 * ow, 1),
        "modell_pick": pick, "wahrscheinlichster_modell_score": pred["most_likely_score"],
        "score_wahrscheinlichkeit_%": round(100 * pred["most_likely_score_prob"], 2),
        "top8_scorelines": top8_from_prediction(pred),
    })
    for tg, og, p in pred["top_scorelines"]:
        score_rows.append({"match": fx["display"], "score": f"{tg}:{og}", "wahrscheinlichkeit_%": round(100 * p, 2)})
summary = pl.DataFrame(preds)
top_scores = pl.DataFrame(score_rows)
print("Deutschland Gruppenspiele")
print(summary)
print("\nTop-Scorelines")
print(top_scores)


## 24. Deutschland Score-Zusammenfassung


In [ ]:
# Zelle 25: Code
# Was diese Zelle macht:
# Verdichtet die Score-Verteilungen und ergänzt Sieg-/Remis-/Niederlage-Wahrscheinlichkeiten für Deutschland.

score_summary_rows = []

for fx in fixtures:
    match_name = fx["display"]

    pred_row = summary.filter(pl.col("match") == match_name).row(0, named=True)
    rows_for_match = top_scores.filter(pl.col("match") == match_name)

    scores = []
    for r in rows_for_match.iter_rows(named=True):
        h, a = map(int, r["score"].split(":"))
        p = float(r["wahrscheinlichkeit_%"]) / 100.0
        scores.append((h, a, p))

    probs = np.array([p for _, _, p in scores], dtype=float)
    probs = probs / probs.sum()

    avg_h = sum(h * w for (h, a, _), w in zip(scores, probs))
    avg_a = sum(a * w for (h, a, _), w in zip(scores, probs))

    top1 = scores[0]
    top2 = scores[1]
    diff_pp = abs(top1[2] - top2[2]) * 100

    practical_pick = (
        f"{top1[0]}:{top1[1]} oder {top2[0]}:{top2[1]}"
        if diff_pp < 0.75
        else f"{top1[0]}:{top1[1]}"
    )

    score_summary_rows.append({
        "match": match_name,
        "win_pick": pred_row["modell_pick"],

        "deutschland_sieg_%": pred_row["deutschland_sieg_%"],
        "remis_%": pred_row["remis_%"],
        "deutschland_niederlage_%": pred_row["gegner_sieg_%"],

        "expected_score": f'{pred_row["deutschland_xg"]:.2f}:{pred_row["gegner_xg"]:.2f}',
        "expected_rounded": f'{round(pred_row["deutschland_xg"])}:{round(pred_row["gegner_xg"])}',
        "top8_weighted": f"{avg_h:.2f}:{avg_a:.2f}",
        "top8_rounded": f"{round(avg_h)}:{round(avg_a)}",
        "practical_score_pick": practical_pick,
        "top1_top2_diff_pp": round(diff_pp, 3),
    })

score_summary = pl.DataFrame(score_summary_rows)

print("Score-Zusammenfassung")
print(score_summary)


## 26. Südkorea Gruppenspiele


In [ ]:
# Zelle 27: Code
# Was diese Zelle macht:
# Erzeugt dieselbe kompakte Prognose für Südkorea gegen Tschechien, Mexiko und Südafrika.

import polars as pl
import numpy as np
from datetime import date

pl.Config.set_tbl_cols(20)
pl.Config.set_tbl_rows(80)
pl.Config.set_fmt_str_lengths(40)

fixtures_korea = [
    {"team": "South Korea", "opponent": "Czech Republic", "display": "Südkorea vs Tschechien", "date": date(2026, 6, 15), "country": "United States"},
    {"team": "South Korea", "opponent": "Mexico", "display": "Südkorea vs Mexiko", "date": date(2026, 6, 20), "country": "United States"},
    {"team": "South Korea", "opponent": "South Africa", "display": "Südkorea vs Südafrika", "date": date(2026, 6, 25), "country": "United States"},
]

DISPLAY_NAMES_KOREA = {
    "Czech Republic": "Tschechien",
    "Mexico": "Mexiko",
    "South Africa": "Südafrika",
}

preds = []
score_rows = []

for fx in fixtures_korea:
    pred = predict_neutral(
        fx["team"],
        fx["opponent"],
        fx["date"],
        country=fx["country"],
    )

    team_win = pred["p_team_win"]
    draw = pred["p_draw"]
    opp_win = pred["p_opponent_win"]

    if team_win > draw and team_win > opp_win:
        pick = "Südkorea gewinnt"
    elif opp_win > team_win and opp_win > draw:
        pick = f'{DISPLAY_NAMES_KOREA.get(fx["opponent"], fx["opponent"])} gewinnt'
    else:
        pick = "Unentschieden"

    preds.append({
        "match": fx["display"],
        "date": fx["date"],
        "suedkorea_xg": round(pred["team_xg_pred"], 3),
        "gegner_xg": round(pred["opponent_xg_pred"], 3),
        "suedkorea_sieg_%": round(100 * team_win, 1),
        "remis_%": round(100 * draw, 1),
        "gegner_sieg_%": round(100 * opp_win, 1),
        "modell_pick": pick,
        "wahrscheinlichster_score": pred["most_likely_score"],
        "score_wahrscheinlichkeit_%": round(100 * pred["most_likely_score_prob"], 2),
    })

    for tg, og, p in pred["top_scorelines"]:
        score_rows.append({
            "match": fx["display"],
            "score": f"{tg}:{og}",
            "wahrscheinlichkeit_%": round(100 * p, 2),
        })

summary_korea = pl.DataFrame(preds)
top_scores_korea = pl.DataFrame(score_rows)

score_summary_rows = []

for fx in fixtures_korea:
    match_name = fx["display"]

    pred_row = summary_korea.filter(pl.col("match") == match_name).row(0, named=True)
    rows_for_match = top_scores_korea.filter(pl.col("match") == match_name)

    scores = []
    for r in rows_for_match.iter_rows(named=True):
        h, a = map(int, r["score"].split(":"))
        p = float(r["wahrscheinlichkeit_%"]) / 100.0
        scores.append((h, a, p))

    probs = np.array([p for _, _, p in scores], dtype=float)
    probs = probs / probs.sum()

    avg_h = sum(h * w for (h, a, _), w in zip(scores, probs))
    avg_a = sum(a * w for (h, a, _), w in zip(scores, probs))

    top1 = scores[0]
    top2 = scores[1]
    diff_pp = abs(top1[2] - top2[2]) * 100

    practical_pick = (
        f"{top1[0]}:{top1[1]} oder {top2[0]}:{top2[1]}"
        if diff_pp < 0.75
        else f"{top1[0]}:{top1[1]}"
    )

    score_summary_rows.append({
        "match": match_name,
        "win_pick": pred_row["modell_pick"],
        "expected_score": f'{pred_row["suedkorea_xg"]:.2f}:{pred_row["gegner_xg"]:.2f}',
        "expected_rounded": f'{round(pred_row["suedkorea_xg"])}:{round(pred_row["gegner_xg"])}',
        "top8_weighted": f"{avg_h:.2f}:{avg_a:.2f}",
        "top8_rounded": f"{round(avg_h)}:{round(avg_a)}",
        "practical_score_pick": practical_pick,
        "top1_top2_diff_pp": round(diff_pp, 3),
    })

score_summary_korea = pl.DataFrame(score_summary_rows)

print("Südkorea Gruppenprognose")
print(summary_korea)

print("\nSüdkorea Top-Scorelines")
print(top_scores_korea)

print("\nSüdkorea Score-Zusammenfassung")
print(score_summary_korea)


## 28. Einzel-Turnierbaum mit allen Spielstats


In [ ]:
# Zelle 29: Code
# EINZEL-VORHERSAGE: kompletter WM-2026-Turnierbaum
# Ausgabe ist kompakt:
# - Gruppenspiele ohne stage/date/city/matchup_original
# - KO-Runden ohne stage/label/date/city/matchup_original
# - KO-"Remis" wird als gleichstand_vor_ne_% angezeigt

import re
import numpy as np
import polars as pl
from pathlib import Path
from datetime import date

pl.Config.set_tbl_cols(50)
pl.Config.set_tbl_rows(220)
pl.Config.set_tbl_width_chars(460)
pl.Config.set_fmt_str_lengths(180)

PREFERRED_VIEW_TEAM = "Germany"

print("=" * 90)
print("EINZEL-VORHERSAGE NUTZT MODELL:")
print("ACTIVE_MODEL_NAME:", globals().get("ACTIVE_MODEL_NAME", "unknown"))
print("predict_match:", getattr(predict_match, "__name__", str(predict_match)))
print("predict_neutral:", getattr(predict_neutral, "__name__", str(predict_neutral)))
print("=" * 90)

REQUIRED_FILES = ["teams.csv", "matches.csv", "tournament_stages.csv", "host_cities.csv"]

candidate_dirs = [
    Path.cwd(),
    Path.cwd() / "archive",
    Path("D:/ml/projects/WeltmeisterKI"),
    Path("D:/ml/projects/WeltmeisterKI/data/raw/wc2026"),
    Path("C:/ml/projects/WeltmeisterKI"),
    Path("C:/Users/samue/Downloads/archive"),
]

CSV_DIR = None
for p in candidate_dirs:
    if all((p / f).exists() for f in REQUIRED_FILES):
        CSV_DIR = p
        break

if CSV_DIR is None:
    raise FileNotFoundError("CSV-Dateien nicht gefunden. Lege sie zum Notebook oder passe CSV_DIR an.")

print("CSV_DIR:", CSV_DIR)

teams = pl.read_csv(CSV_DIR / "teams.csv")
matches = pl.read_csv(CSV_DIR / "matches.csv")
stages = pl.read_csv(CSV_DIR / "tournament_stages.csv")
cities = pl.read_csv(CSV_DIR / "host_cities.csv")

PLACEHOLDER_REPLACEMENTS = {
    "Winner UEFA Playoff D": "Czech Republic",
    "Winner UEFA Playoff A": "Bosnia and Herzegovina",
    "Winner UEFA Playoff C": "Turkey",
    "Winner UEFA Playoff B": "Sweden",
    "Winner FIFA Playoff 2": "Iraq",
    "Winner FIFA Playoff 1": "DR Congo",
}

teams = teams.with_columns(
    pl.when(pl.col("team_name").is_in(list(PLACEHOLDER_REPLACEMENTS.keys())))
    .then(pl.col("team_name").replace(PLACEHOLDER_REPLACEMENTS))
    .otherwise(pl.col("team_name"))
    .alias("team_name")
)

matches = matches.with_columns(
    pl.when((pl.col("match_number") == 100) & (pl.col("match_label").str.contains("W100")))
    .then(pl.lit("W95 vs W96"))
    .otherwise(pl.col("match_label"))
    .alias("match_label")
)

team_id_to_name = dict(teams.select(["id", "team_name"]).iter_rows())
team_to_group = dict(teams.select(["team_name", "group_letter"]).iter_rows())
stage_id_to_name = dict(stages.select(["id", "stage_name"]).iter_rows())
city_id_to_country = dict(cities.select(["id", "country"]).iter_rows())
city_id_to_city = dict(cities.select(["id", "city_name"]).iter_rows())

group_stage_id = next(
    sid for sid, sname in stage_id_to_name.items()
    if "group" in str(sname).lower()
)

group_matches_single = matches.filter(pl.col("stage_id") == group_stage_id).sort("match_number").to_dicts()
ko_matches_single = matches.filter(pl.col("stage_id") != group_stage_id).sort("match_number").to_dicts()
all_teams_single = sorted(team_to_group.keys())

print("Teams:", len(all_teams_single))
print("Group matches:", len(group_matches_single))
print("KO matches:", len(ko_matches_single))

HOST_COUNTRY_TO_TEAM = {
    "United States": "USA",
    "USA": "USA",
    "Mexico": "Mexico",
    "Canada": "Canada",
}

def parse_match_date_single(x):
    return date.fromisoformat(str(x)[:10])

def copy_grid(grid):
    g = np.asarray(grid, dtype=float).copy()
    g = np.clip(g, 1e-12, None)
    return g / g.sum()

def outcome_probs_from_grid_single(grid):
    return {
        "p_home_win": float(np.tril(grid, -1).sum()),
        "p_draw": float(np.trace(grid)),
        "p_away_win": float(np.triu(grid, 1).sum()),
    }

def top_scorelines_for_view_single(grid, view_is_home=True, top_n=8):
    if grid is None:
        return None
    g = np.asarray(grid, dtype=float)
    rows = []
    for h in range(g.shape[0]):
        for a in range(g.shape[1]):
            score = f"{h}:{a}" if view_is_home else f"{a}:{h}"
            rows.append((score, float(g[h, a])))
    rows.sort(key=lambda x: x[1], reverse=True)
    return ", ".join([f"{score} ({100*p:.1f}%)" for score, p in rows[:top_n]])


def ko_advance_probs_from_grid_single(grid):
    probs = outcome_probs_from_grid_single(grid)
    p_home = probs["p_home_win"]
    p_draw = probs["p_draw"]
    p_away = probs["p_away_win"]

    pen_home = p_home / max(p_home + p_away, 1e-9)

    return {
        "p_home_advance": float(p_home + p_draw * pen_home),
        "p_away_advance": float(p_away + p_draw * (1.0 - pen_home)),
    }

single_dist_cache = {}

def predict_distribution_single(home, away, match_date, country, max_goals=10):
    key = (home, away, str(match_date), country, globals().get("ACTIVE_MODEL_NAME", "active"))
    if key in single_dist_cache:
        return single_dist_cache[key]

    host_team = HOST_COUNTRY_TO_TEAM.get(country)

    if host_team == home:
        pred = predict_match(
            home, away, match_date,
            tournament="FIFA World Cup",
            country=country,
            neutral=False,
            max_goals=max_goals,
        )
        grid = copy_grid(pred["score_grid"])
        home_xg = float(pred.get("home_xg_pred", pred.get("home_xg")))
        away_xg = float(pred.get("away_xg_pred", pred.get("away_xg")))

    elif host_team == away:
        pred_rev = predict_match(
            away, home, match_date,
            tournament="FIFA World Cup",
            country=country,
            neutral=False,
            max_goals=max_goals,
        )
        grid = copy_grid(pred_rev["score_grid"]).T
        home_xg = float(pred_rev.get("away_xg_pred", pred_rev.get("away_xg")))
        away_xg = float(pred_rev.get("home_xg_pred", pred_rev.get("home_xg")))

    else:
        pred = predict_neutral(
            home, away, match_date,
            tournament="FIFA World Cup",
            country=country,
            max_goals=max_goals,
        )
        grid = copy_grid(pred["score_grid"])
        home_xg = float(pred.get("team_xg_pred", pred.get("home_xg_pred")))
        away_xg = float(pred.get("opponent_xg_pred", pred.get("away_xg_pred")))

    h, a = np.unravel_index(np.argmax(grid), grid.shape)

    out = {
        "home": home,
        "away": away,
        "date": match_date,
        "country": country,
        "grid": grid,
        "home_xg": home_xg,
        "away_xg": away_xg,
        "most_likely_home_goals": int(h),
        "most_likely_away_goals": int(a),
        "most_likely_score_prob": float(grid[h, a]),
        **outcome_probs_from_grid_single(grid),
        **ko_advance_probs_from_grid_single(grid),
    }

    single_dist_cache[key] = out
    return out

def deterministic_match_single(home, away, match_date, country, knockout=False):
    dist = predict_distribution_single(home, away, match_date, country)

    hs = dist["most_likely_home_goals"]
    aw = dist["most_likely_away_goals"]

    tiebreak = ""

    if hs > aw:
        winner, loser = home, away
    elif aw > hs:
        winner, loser = away, home
    else:
        if knockout:
            if dist["p_home_advance"] >= dist["p_away_advance"]:
                winner, loser = home, away
            else:
                winner, loser = away, home
            tiebreak = " n.E."
        else:
            winner, loser = None, None

    return {
        "home": home,
        "away": away,
        "home_score": hs,
        "away_score": aw,
        "score": f"{hs}:{aw}{tiebreak}",
        "winner": winner,
        "loser": loser,
        "home_xg": dist["home_xg"],
        "away_xg": dist["away_xg"],
        "p_home_win": dist["p_home_win"],
        "p_draw": dist["p_draw"],
        "p_away_win": dist["p_away_win"],
        "p_home_advance": dist["p_home_advance"],
        "p_away_advance": dist["p_away_advance"],
        "score_grid": dist["grid"],
        "score_prob_%": round(100 * dist["most_likely_score_prob"], 2),
        "status": "Prognose",
    }

def format_view_record(base, res, knockout=False):
    home = res["home"]
    away = res["away"]

    if PREFERRED_VIEW_TEAM in (home, away):
        view = PREFERRED_VIEW_TEAM
    else:
        view = home

    if view == home:
        opp = away
        view_score = res["score"]
        view_xg = res["home_xg"]
        opp_xg = res["away_xg"]
        p_win = res["p_home_win"]
        p_loss = res["p_away_win"]
        p_advance = res["p_home_advance"]
    else:
        opp = home
        raw_score = str(res["score"])
        pen = " n.E." if "n.E." in raw_score else ""
        clean = raw_score.replace(" n.E.", "")
        hs, aw = map(int, clean.split(":"))

        view_score = f"{aw}:{hs}{pen}"
        view_xg = res["away_xg"]
        opp_xg = res["home_xg"]
        p_win = res["p_away_win"]
        p_loss = res["p_home_win"]
        p_advance = res["p_away_advance"]

    out = dict(base)
    out.update({
        "sicht_matchup": f"{view} vs {opp}",
        "sicht_team": view,
        "gegner": opp,
        "sicht_score": view_score,
        "winner": res["winner"],
        "loser": res["loser"],
        "sicht_xg": round(view_xg, 3),
        "gegner_xg": round(opp_xg, 3),
        "sicht_sieg_%": round(100 * p_win, 1),
        "sicht_niederlage_%": round(100 * p_loss, 1),
        "score_prob_%": res["score_prob_%"],
        "top8_scorelines": top_scorelines_for_view_single(res.get("score_grid"), view_is_home=(view == home)) if res.get("score_grid") is not None else None,
        "status": res.get("status", "Prognose"),
    })

    if knockout:
        out["gleichstand_vor_ne_%"] = round(100 * res["p_draw"], 1)
        out["sicht_weiterkommen_%"] = round(100 * p_advance, 1)
    else:
        out["remis_%"] = round(100 * res["p_draw"], 1)

    return out

def empty_table_row_single(team):
    return {
        "team": team,
        "group": team_to_group[team],
        "mp": 0,
        "w": 0,
        "d": 0,
        "l": 0,
        "gf": 0,
        "ga": 0,
        "gd": 0,
        "pts": 0,
    }

def add_group_result_single(table, home, away, hs, aw):
    table[home]["mp"] += 1
    table[away]["mp"] += 1

    table[home]["gf"] += hs
    table[home]["ga"] += aw
    table[away]["gf"] += aw
    table[away]["ga"] += hs

    table[home]["gd"] = table[home]["gf"] - table[home]["ga"]
    table[away]["gd"] = table[away]["gf"] - table[away]["ga"]

    if hs > aw:
        table[home]["w"] += 1
        table[away]["l"] += 1
        table[home]["pts"] += 3
    elif aw > hs:
        table[away]["w"] += 1
        table[home]["l"] += 1
        table[away]["pts"] += 3
    else:
        table[home]["d"] += 1
        table[away]["d"] += 1
        table[home]["pts"] += 1
        table[away]["pts"] += 1

def team_tiebreak_strength(team):
    score = 0.0

    try:
        score += float(state["elo"][norm_team(team)])
    except Exception:
        pass

    try:
        rank, points, conf = fifa_before(norm_team(team), date(2026, 6, 1))
        if not np.isnan(rank):
            score += 250.0 - float(rank)
        if not np.isnan(points):
            score += float(points) / 10.0
    except Exception:
        pass

    return score

def rank_group_rows_single(rows):
    return sorted(
        rows,
        key=lambda r: (
            r["pts"],
            r["gd"],
            r["gf"],
            r["w"],
            team_tiebreak_strength(r["team"]),
        ),
        reverse=True,
    )

def split_match_label_single(label):
    return [x.strip() for x in re.split(r"\s+vs\s+", str(label))]

third_slot_pattern = re.compile(r"^3([A-L]+)$")

def collect_third_slots_single():
    slots = []
    for m in ko_matches_single:
        for p in split_match_label_single(m["match_label"]):
            if third_slot_pattern.fullmatch(p):
                slots.append(p)
    return list(dict.fromkeys(slots))

THIRD_SLOTS_SINGLE = collect_third_slots_single()

def allocate_third_place_slots_single(best_thirds):
    third_by_group = {r["group"]: r for r in best_thirds}
    available_groups = set(third_by_group.keys())
    third_rank_order = [r["group"] for r in best_thirds]
    allowed = {slot: set(slot[1:]) for slot in THIRD_SLOTS_SINGLE}
    ordered_slots = sorted(THIRD_SLOTS_SINGLE, key=lambda s: len(allowed[s] & available_groups))

    def rec(i, remaining, assignment):
        if i == len(ordered_slots):
            return assignment

        slot = ordered_slots[i]
        candidates = [
            g for g in third_rank_order
            if g in remaining and g in allowed[slot]
        ]

        for g in candidates:
            nxt = dict(assignment)
            nxt[slot] = third_by_group[g]["team"]
            result = rec(i + 1, remaining - {g}, nxt)
            if result is not None:
                return result

        return None

    assignment = rec(0, available_groups, {})
    if assignment is None:
        raise RuntimeError(f"Kann Third-place Slots nicht zuordnen: {sorted(available_groups)}")

    return assignment

def resolve_slot_single(slot, qualifiers, match_results):
    slot = str(slot).strip()

    if re.fullmatch(r"[12][A-L]", slot):
        return qualifiers[slot]

    if third_slot_pattern.fullmatch(slot):
        return qualifiers[slot]

    if re.fullmatch(r"W\d+", slot):
        return match_results[int(slot[1:])]["winner"]

    if re.fullmatch(r"RU\d+", slot):
        return match_results[int(slot[2:])]["loser"]

    raise RuntimeError(f"Unbekannter KO-Slot: {slot}")

table = {team: empty_table_row_single(team) for team in all_teams_single}
group_match_records = []

for m in group_matches_single:
    mn = int(m["match_number"])
    home = team_id_to_name[int(m["home_team_id"])]
    away = team_id_to_name[int(m["away_team_id"])]
    d = parse_match_date_single(m["kickoff_at"])
    country = city_id_to_country[int(m["city_id"])]

    fixed = get_played_match_result(mn, home, away) if "get_played_match_result" in globals() else None
    if fixed is not None:
        res = played_result_as_res(fixed, home, away, d, country, knockout=False, distribution_func=predict_distribution_single)
    else:
        res = deterministic_match_single(home, away, d, country, knockout=False)

    add_group_result_single(table, home, away, res["home_score"], res["away_score"])

    group_match_records.append(format_view_record({
        "match": mn,
        "group": team_to_group[home],
    }, res, knockout=False))

group_rankings = {}
qualifiers = {}

for g in sorted(set(team_to_group.values())):
    ranked = rank_group_rows_single([dict(v) for v in table.values() if v["group"] == g])
    group_rankings[g] = ranked
    qualifiers[f"1{g}"] = ranked[0]["team"]
    qualifiers[f"2{g}"] = ranked[1]["team"]

thirds = [group_rankings[g][2] for g in sorted(group_rankings.keys())]
best_thirds = rank_group_rows_single(thirds)[:8]
qualifiers.update(allocate_third_place_slots_single(best_thirds))

match_results = {}
ko_records = []

for m in ko_matches_single:
    mn = int(m["match_number"])
    stage = stage_id_to_name[int(m["stage_id"])]
    label = str(m["match_label"])
    parts = split_match_label_single(label)

    home = resolve_slot_single(parts[0], qualifiers, match_results)
    away = resolve_slot_single(parts[1], qualifiers, match_results)

    d = parse_match_date_single(m["kickoff_at"])
    country = city_id_to_country[int(m["city_id"])]

    res = deterministic_match_single(home, away, d, country, knockout=True)
    match_results[mn] = res

    ko_records.append(format_view_record({
        "match": mn,
        "runde": stage,
    }, res, knockout=True))

single_group_matches = pl.DataFrame(group_match_records).sort("match")

single_group_tables_rows = []
single_group_winners_rows = []
single_best_thirds_rows = []

for g in sorted(group_rankings.keys()):
    for pos, r in enumerate(group_rankings[g], start=1):
        single_group_tables_rows.append({
            "group": g,
            "pos": pos,
            "team": r["team"],
            "pts": r["pts"],
            "mp": r["mp"],
            "w": r["w"],
            "d": r["d"],
            "l": r["l"],
            "gf": r["gf"],
            "ga": r["ga"],
            "gd": r["gd"],
        })

    single_group_winners_rows.append({
        "group": g,
        "sieger": group_rankings[g][0]["team"],
        "sieger_pts": group_rankings[g][0]["pts"],
        "zweiter": group_rankings[g][1]["team"],
        "zweiter_pts": group_rankings[g][1]["pts"],
        "dritter": group_rankings[g][2]["team"],
        "dritter_pts": group_rankings[g][2]["pts"],
    })

for pos, r in enumerate(best_thirds, start=1):
    single_best_thirds_rows.append({
        "best_third_rank": pos,
        "group": r["group"],
        "team": r["team"],
        "pts": r["pts"],
        "gd": r["gd"],
        "gf": r["gf"],
    })

single_group_tables = pl.DataFrame(single_group_tables_rows).sort(["group", "pos"])
single_group_winners = pl.DataFrame(single_group_winners_rows).sort("group")
single_best_thirds = pl.DataFrame(single_best_thirds_rows)
single_ko_tree = pl.DataFrame(ko_records).sort("match")

final_rec = next((r for r in ko_records if str(r["runde"]).lower() == "final"), None)
third_rec = next((r for r in ko_records if "third" in str(r["runde"]).lower()), None)

single_medals = pl.DataFrame([{
    "gold": final_rec["winner"] if final_rec else None,
    "silver": final_rec["loser"] if final_rec else None,
    "bronze": third_rec["winner"] if third_rec else None,
    "fourth": third_rec["loser"] if third_rec else None,
}])

GAME_COLS_GROUP = [
    "match",
    "group",
    "sicht_matchup",
    "sicht_score",
    "sicht_sieg_%",
    "remis_%",
    "sicht_niederlage_%",
    "sicht_xg",
    "gegner_xg",
    "score_prob_%",
    "top8_scorelines",
    "status",
]

GAME_COLS_KO = [
    "match",
    "sicht_matchup",
    "sicht_score",
    "winner",
    "loser",
    "sicht_sieg_%",
    "sicht_niederlage_%",
    "gleichstand_vor_ne_%",
    "sicht_weiterkommen_%",
    "sicht_xg",
    "gegner_xg",
    "score_prob_%",
    "top8_scorelines",
    "status",
]

print("\n=== Alle Gruppenspiele: konkrete Einzel-Vorhersage ===")
print(single_group_matches.select(GAME_COLS_GROUP))

print("\n=== Gruppentabellen ===")
print(single_group_tables)

print("\n=== Gruppensieger / Gruppenzweite / Gruppendritte ===")
print(single_group_winners)

print("\n=== Beste Gruppendritte, die weiterkommen ===")
print(single_best_thirds)

print("\n=== KO-Baum: konkrete Einzel-Vorhersage ===")
for stage in ["Round of 32", "Round of 16", "Quarterfinals", "Semifinals", "Third Place Playoff", "Final"]:
    part = single_ko_tree.filter(pl.col("runde") == stage)
    if part.height == 0:
        continue

    print(f"\n--- {stage} ---")
    print(part.select(GAME_COLS_KO))

print("\n=== Medaillen: konkrete Einzel-Vorhersage ===")
print(single_medals)

print("\n=== Deutschland / Suedkorea im Einzel-Turnier ===")
for team in ["Germany", "South Korea"]:
    team_group = single_group_tables.filter(pl.col("team") == team)
    team_ko = single_ko_tree.filter(
        (pl.col("sicht_matchup").str.contains(team))
        | (pl.col("winner") == team)
        | (pl.col("loser") == team)
    )

    print(f"\n{team} Gruppentabelle:")
    print(team_group)

    print(f"\n{team} KO-Spiele:")
    if team_ko.height:
        print(team_ko.select([
            "runde",
            "sicht_matchup",
            "sicht_score",
            "winner",
            "loser",
            "sicht_sieg_%",
            "sicht_niederlage_%",
            "gleichstand_vor_ne_%",
            "sicht_weiterkommen_%",
        ]))
    else:
        print("Keine KO-Spiele im konkreten Einzelpfad.")


## 30. Monte Carlo WM 2026


In [ ]:
# Zelle 31: Code
# Was diese Zelle macht:
# Simuliert die komplette WM viele Male mit dem aktiven Modell. Ergebnis: Titelchancen, 16telfinalisten, Exit-Runden und häufigste Exit-Gegner.

import re
import math
import time
import numpy as np
import polars as pl
from pathlib import Path
from collections import defaultdict, Counter
from datetime import date

# Monte Carlo WM 2026
# Nutzt den aktiven Prediction Helper aus predict_match/predict_neutral: v2 oder Hybrid, je nach Auswahl-Zelle.
# Voraussetzung:
# - Prediction Helper v2 wurde ausgeführt
# - predict_match(...)
# - predict_neutral(...)
# - score_grid im Output vorhanden

N_SIMS = 50_000
RNG_SEED = 42
MAX_GOALS = 10

rng = np.random.default_rng(RNG_SEED)

pl.Config.set_tbl_cols(35)
pl.Config.set_tbl_rows(100)
pl.Config.set_tbl_width_chars(340)
pl.Config.set_fmt_str_lengths(160)

required_prediction_helpers = ["predict_match", "predict_neutral"]
missing_prediction_helpers = [x for x in required_prediction_helpers if x not in globals()]

if missing_prediction_helpers:
    raise RuntimeError(
        "Bitte erst die Prediction-Helper-v2-Zelle ausführen. "
        f"Fehlt aktuell: {missing_prediction_helpers}"
    )

print("Monte Carlo nutzt Prediction Helper:")
print("predict_match:", predict_match)
print("predict_neutral:", predict_neutral)

if "calibration" in globals():
    print("Calibration:", calibration)

# 1. CSV-Ordner finden

REQUIRED_FILES = ["teams.csv", "matches.csv", "tournament_stages.csv", "host_cities.csv"]

candidate_dirs = [
    Path.cwd(),
    Path.cwd() / "archive",
    Path("D:/ml/projects/WeltmeisterKI"),
    Path("D:/ml/projects/WeltmeisterKI/data/raw/wc2026"),
    Path("C:/ml/projects/WeltmeisterKI"),
    Path("C:/Users/samue/Downloads/archive"),
]

CSV_DIR = None
for p in candidate_dirs:
    if all((p / f).exists() for f in REQUIRED_FILES):
        CSV_DIR = p
        break

if CSV_DIR is None:
    raise FileNotFoundError(
        "Ich finde teams.csv/matches.csv/tournament_stages.csv/host_cities.csv nicht. "
        "Lege sie in denselben Ordner wie das Notebook oder passe CSV_DIR manuell an."
    )

print("CSV_DIR:", CSV_DIR)

teams = pl.read_csv(CSV_DIR / "teams.csv")
matches = pl.read_csv(CSV_DIR / "matches.csv")
stages = pl.read_csv(CSV_DIR / "tournament_stages.csv")
cities = pl.read_csv(CSV_DIR / "host_cities.csv")

# 2. Placeholder ersetzen

PLACEHOLDER_REPLACEMENTS = {
    "Winner UEFA Playoff D": "Czech Republic",
    "Winner UEFA Playoff A": "Bosnia and Herzegovina",
    "Winner UEFA Playoff C": "Turkey",
    "Winner UEFA Playoff B": "Sweden",
    "Winner FIFA Playoff 2": "Iraq",
    "Winner FIFA Playoff 1": "DR Congo",
}

teams = teams.with_columns(
    pl.when(pl.col("team_name").is_in(list(PLACEHOLDER_REPLACEMENTS.keys())))
    .then(pl.col("team_name").replace(PLACEHOLDER_REPLACEMENTS))
    .otherwise(pl.col("team_name"))
    .alias("team_name")
)

if "is_placeholder" in teams.columns:
    teams = teams.with_columns(
        pl.when(pl.col("team_name").is_in(list(PLACEHOLDER_REPLACEMENTS.values())))
        .then(False)
        .otherwise(pl.col("is_placeholder"))
        .alias("is_placeholder")
    )

# Patch für fehlerhafte self-reference in alter CSV.
matches = matches.with_columns(
    pl.when((pl.col("match_number") == 100) & (pl.col("match_label").str.contains("W100")))
    .then(pl.lit("W95 vs W96"))
    .otherwise(pl.col("match_label"))
    .alias("match_label")
)

# 3. Lookups

team_id_to_name = dict(teams.select(["id", "team_name"]).iter_rows())
team_to_group = dict(teams.select(["team_name", "group_letter"]).iter_rows())

stage_id_to_name = dict(stages.select(["id", "stage_name"]).iter_rows())
city_id_to_country = dict(cities.select(["id", "country"]).iter_rows())
city_id_to_city = dict(cities.select(["id", "city_name"]).iter_rows())

group_stage_id = None
for sid, sname in stage_id_to_name.items():
    if "group" in str(sname).lower():
        group_stage_id = sid
        break

if group_stage_id is None:
    raise RuntimeError("Group-Stage-ID nicht gefunden.")

group_matches = (
    matches
    .filter(pl.col("stage_id") == group_stage_id)
    .sort("match_number")
    .to_dicts()
)

ko_matches = (
    matches
    .filter(pl.col("stage_id") != group_stage_id)
    .sort("match_number")
    .to_dicts()
)

all_teams = sorted(team_to_group.keys())

print("Teams:", len(all_teams))
print("Group matches:", len(group_matches))
print("KO matches:", len(ko_matches))

# 4. Aktive Match-Wahrscheinlichkeiten

HOST_COUNTRY_TO_TEAM = {
    "United States": "USA",
    "USA": "USA",
    "Mexico": "Mexico",
    "Canada": "Canada",
}

def parse_match_date(x):
    return date.fromisoformat(str(x)[:10])

def outcome_probs_from_grid(grid):
    p_home = float(np.tril(grid, -1).sum())
    p_draw = float(np.trace(grid))
    p_away = float(np.triu(grid, 1).sum())

    probs = np.array([p_home, p_draw, p_away], dtype=float)
    probs = np.clip(probs, 1e-9, 1.0)
    probs = probs / probs.sum()

    return float(probs[0]), float(probs[1]), float(probs[2])

dist_cache = {}

def _copy_grid_as_float(grid):
    g = np.asarray(grid, dtype=float).copy()
    g = np.clip(g, 0.0, 1.0)
    g = g / g.sum()
    return g

def predict_match_distribution(home, away, match_date, country):
    """
    Aktive Modell-Version:
    Gibt eine komplette kalibrierte Score-Verteilung in Home/Away-Reihenfolge zurück.

    Wichtig:
    - Neutral: predict_neutral(home, away) -> score_grid ist home/away aus Sicht der Argumente.
    - Host home: predict_match(home, away, neutral=False)
    - Host away: predict_match(away, home, neutral=False), dann Grid transponieren.
    """
    key = (home, away, str(match_date), country, str(globals().get("ACTIVE_MODEL_NAME", "active_model")), str(globals().get("PLAYED_MATCHES_VERSION", "no_played")))
    if key in dist_cache:
        return dist_cache[key]

    host_team = HOST_COUNTRY_TO_TEAM.get(country)

    if host_team == home:
        pred = predict_match(
            home,
            away,
            match_date,
            country=country,
            neutral=False,
            max_goals=MAX_GOALS,
        )

        grid = _copy_grid_as_float(pred["score_grid"])
        home_xg = float(pred["home_xg"])
        away_xg = float(pred["away_xg"])

    elif host_team == away:
        pred_reversed = predict_match(
            away,
            home,
            match_date,
            country=country,
            neutral=False,
            max_goals=MAX_GOALS,
        )

        # pred_reversed Grid ist away/home. Wir brauchen home/away.
        grid = _copy_grid_as_float(pred_reversed["score_grid"]).T
        home_xg = float(pred_reversed["away_xg"])
        away_xg = float(pred_reversed["home_xg"])

    else:
        pred = predict_neutral(
            home,
            away,
            match_date,
            country=country,
            max_goals=MAX_GOALS,
        )

        # predict_neutral gibt Grid in team/opponent-Reihenfolge zurück,
        # also hier home/away.
        grid = _copy_grid_as_float(pred["score_grid"])
        home_xg = float(pred["team_xg_pred"])
        away_xg = float(pred["opponent_xg_pred"])

    p_home_win, p_draw, p_away_win = outcome_probs_from_grid(grid)

    top_i, top_j = np.unravel_index(np.argmax(grid), grid.shape)

    dist = {
        "home": home,
        "away": away,
        "home_xg": home_xg,
        "away_xg": away_xg,
        "grid": grid,
        "flat": grid.reshape(-1),
        "p_home_win": p_home_win,
        "p_draw": p_draw,
        "p_away_win": p_away_win,
        "most_likely_score": f"{top_i}:{top_j}",
        "most_likely_score_prob": float(grid[top_i, top_j]),
    }

    dist_cache[key] = dist
    return dist

def sample_score_from_dist(dist):
    grid = dist["grid"]
    flat = dist["flat"]

    idx = rng.choice(flat.size, p=flat)
    h = idx // grid.shape[1]
    a = idx % grid.shape[1]

    return int(h), int(a)

def simulate_match(home, away, match_date, country, knockout=False):
    dist = predict_match_distribution(home, away, match_date, country)
    hs, aw = sample_score_from_dist(dist)

    tiebreak = ""

    if hs > aw:
        winner, loser = home, away
    elif aw > hs:
        winner, loser = away, home
    else:
        if knockout:
            # Bei KO-Remis: Elfmeterschießen.
            # Stärkeindikator: kalibrierte non-draw Win-Wahrscheinlichkeiten.
            p_home = dist["p_home_win"]
            p_away = dist["p_away_win"]

            if p_home + p_away <= 0:
                pen_home_prob = 0.5
            else:
                pen_home_prob = p_home / (p_home + p_away)

            if rng.random() < pen_home_prob:
                winner, loser = home, away
            else:
                winner, loser = away, home

            tiebreak = " n.E."
        else:
            winner, loser = None, None

    return {
        "home": home,
        "away": away,
        "home_score": hs,
        "away_score": aw,
        "score": f"{hs}:{aw}{tiebreak}",
        "winner": winner,
        "loser": loser,
        "home_xg": dist["home_xg"],
        "away_xg": dist["away_xg"],
        "p_home_win": dist["p_home_win"],
        "p_draw": dist["p_draw"],
        "p_away_win": dist["p_away_win"],
        "most_likely_score": dist["most_likely_score"],
        "most_likely_score_prob": dist["most_likely_score_prob"],
    }

# 5. Gruppenphase

def empty_table_row(team):
    return {
        "team": team,
        "group": team_to_group[team],
        "mp": 0,
        "w": 0,
        "d": 0,
        "l": 0,
        "gf": 0,
        "ga": 0,
        "gd": 0,
        "pts": 0,
    }

def add_group_result(table, home, away, hs, aw):
    table[home]["mp"] += 1
    table[away]["mp"] += 1

    table[home]["gf"] += hs
    table[home]["ga"] += aw
    table[away]["gf"] += aw
    table[away]["ga"] += hs

    table[home]["gd"] = table[home]["gf"] - table[home]["ga"]
    table[away]["gd"] = table[away]["gf"] - table[away]["ga"]

    if hs > aw:
        table[home]["w"] += 1
        table[away]["l"] += 1
        table[home]["pts"] += 3
    elif aw > hs:
        table[away]["w"] += 1
        table[home]["l"] += 1
        table[away]["pts"] += 3
    else:
        table[home]["d"] += 1
        table[away]["d"] += 1
        table[home]["pts"] += 1
        table[away]["pts"] += 1

def rank_group_rows(rows):
    # Vereinfachter Tie-break:
    # Punkte, Tordifferenz, Tore, Siege, dann Random-Tiebreak.
    return sorted(
        rows,
        key=lambda r: (
            r["pts"],
            r["gd"],
            r["gf"],
            r["w"],
            rng.random(),
        ),
        reverse=True
    )

# 6. Third-place Slots

def split_match_label(label):
    return [x.strip() for x in re.split(r"\s+vs\s+", str(label))]

third_slot_pattern = re.compile(r"^3([A-L]+)$")

def collect_third_slots():
    slots = []
    for m in ko_matches:
        parts = split_match_label(m["match_label"])
        for p in parts:
            if third_slot_pattern.fullmatch(p):
                slots.append(p)

    seen = set()
    out = []
    for s in slots:
        if s not in seen:
            seen.add(s)
            out.append(s)

    return out

THIRD_SLOTS = collect_third_slots()
print("Third-place slots:", THIRD_SLOTS)

def allocate_third_place_slots(best_thirds):
    third_by_group = {r["group"]: r for r in best_thirds}
    available_groups = set(third_by_group.keys())
    third_rank_order = [r["group"] for r in best_thirds]

    allowed = {
        slot: set(slot[1:])
        for slot in THIRD_SLOTS
    }

    ordered_slots = sorted(
        THIRD_SLOTS,
        key=lambda s: len(allowed[s] & available_groups)
    )

    def rec(i, remaining, assignment):
        if i == len(ordered_slots):
            return assignment

        slot = ordered_slots[i]
        candidates = [
            g for g in third_rank_order
            if g in remaining and g in allowed[slot]
        ]

        for g in candidates:
            new_assignment = dict(assignment)
            new_assignment[slot] = third_by_group[g]["team"]
            result = rec(i + 1, remaining - {g}, new_assignment)
            if result is not None:
                return result

        return None

    assignment = rec(0, available_groups, {})

    if assignment is None:
        raise RuntimeError(
            f"Kann Third-place Slots nicht zuordnen. "
            f"Best third groups: {sorted(available_groups)}, Slots: {THIRD_SLOTS}"
        )

    return assignment

# 7. KO Resolver

def resolve_slot(slot, qualifiers, match_results):
    slot = str(slot).strip()

    if re.fullmatch(r"[12][A-L]", slot):
        return qualifiers[slot]

    if third_slot_pattern.fullmatch(slot):
        return qualifiers[slot]

    if re.fullmatch(r"W\d+", slot):
        num = int(slot[1:])
        return match_results[num]["winner"]

    if re.fullmatch(r"RU\d+", slot):
        num = int(slot[2:])
        return match_results[num]["loser"]

    raise RuntimeError(f"Unbekannter KO-Slot: {slot}")

# 8. Ein Turnier simulieren

def simulate_tournament_once():
    table = {team: empty_table_row(team) for team in all_teams}
    group_match_records = []

    for m in group_matches:
        home = team_id_to_name[int(m["home_team_id"])]
        away = team_id_to_name[int(m["away_team_id"])]
        d = parse_match_date(m["kickoff_at"])
        country = city_id_to_country[int(m["city_id"])]

        fixed = get_played_match_result(int(m["match_number"]), home, away) if "get_played_match_result" in globals() else None
        if fixed is not None:
            res = played_result_as_res(fixed, home, away, d, country, knockout=False, distribution_func=predict_match_distribution)
        else:
            res = simulate_match(home, away, d, country, knockout=False)

        add_group_result(table, home, away, res["home_score"], res["away_score"])

        group_match_records.append({
            "match_number": int(m["match_number"]),
            "group": team_to_group[home],
            "home": home,
            "away": away,
            "score": res["score"],
            "status": res.get("status", "Prognose"),
        })

    group_rankings = {}
    qualifiers = {}

    for g in sorted(set(team_to_group.values())):
        rows = [dict(v) for v in table.values() if v["group"] == g]
        ranked = rank_group_rows(rows)
        group_rankings[g] = ranked

        qualifiers[f"1{g}"] = ranked[0]["team"]
        qualifiers[f"2{g}"] = ranked[1]["team"]

    thirds = [group_rankings[g][2] for g in sorted(group_rankings.keys())]
    best_thirds = rank_group_rows(thirds)[:8]
    third_assignment = allocate_third_place_slots(best_thirds)
    qualifiers.update(third_assignment)

    round32_teams = set()
    for key, team in qualifiers.items():
        if re.fullmatch(r"[123][A-L]+", key):
            round32_teams.add(team)

    match_results = {}
    knockout_records = []

    for m in ko_matches:
        mn = int(m["match_number"])
        stage = stage_id_to_name[int(m["stage_id"])]
        parts = split_match_label(m["match_label"])

        if len(parts) != 2:
            raise RuntimeError(f"Kann match_label nicht splitten: {m['match_label']}")

        home = resolve_slot(parts[0], qualifiers, match_results)
        away = resolve_slot(parts[1], qualifiers, match_results)

        d = parse_match_date(m["kickoff_at"])
        country = city_id_to_country[int(m["city_id"])]

        res = simulate_match(home, away, d, country, knockout=True)
        match_results[mn] = res

        knockout_records.append({
            "match_number": mn,
            "stage": stage,
            "home": home,
            "away": away,
            "score": res["score"],
            "winner": res["winner"],
            "loser": res["loser"],
        })

    final_match = None
    third_place_match = None

    for rec in knockout_records:
        stage_lower = rec["stage"].lower()
        if stage_lower == "final":
            final_match = rec
        elif "third" in stage_lower:
            third_place_match = rec

    if final_match is None:
        raise RuntimeError("Finale nicht gefunden.")

    gold = final_match["winner"]
    silver = final_match["loser"]

    if third_place_match is not None:
        bronze = third_place_match["winner"]
        fourth = third_place_match["loser"]
    else:
        bronze = None
        fourth = None

    finish_status = {team: "Group Stage" for team in all_teams}
    exit_opponent = {team: None for team in all_teams}

    for team in round32_teams:
        finish_status[team] = "Round of 32"

    for rec in knockout_records:
        loser = rec["loser"]
        winner = rec["winner"]
        stage = rec["stage"]

        if stage == "Final":
            finish_status[loser] = "Final / Silver"
            exit_opponent[loser] = winner
        elif "Third" in stage:
            finish_status[loser] = "Fourth Place"
            exit_opponent[loser] = winner
            finish_status[winner] = "Third Place"
        else:
            finish_status[loser] = stage
            exit_opponent[loser] = winner

    finish_status[gold] = "Champion"
    exit_opponent[gold] = None

    return {
        "gold": gold,
        "silver": silver,
        "bronze": bronze,
        "fourth": fourth,
        "round32_teams": round32_teams,
        "group_winners": {g: rows[0]["team"] for g, rows in group_rankings.items()},
        "group_runners_up": {g: rows[1]["team"] for g, rows in group_rankings.items()},
        "best_thirds": [r["team"] for r in best_thirds],
        "finish_status": finish_status,
        "exit_opponent": exit_opponent,
        "final_pair": tuple(sorted([gold, silver])),
    }

# 9. Monte Carlo

title_counts = Counter()
silver_counts = Counter()
bronze_counts = Counter()
fourth_counts = Counter()
round32_counts = Counter()
group_winner_counts = Counter()
best_third_counts = Counter()
finish_counts = defaultdict(Counter)
exit_vs_counts = defaultdict(Counter)
final_pair_counts = Counter()

start = time.time()

for i in range(1, N_SIMS + 1):
    sim = simulate_tournament_once()

    title_counts[sim["gold"]] += 1
    silver_counts[sim["silver"]] += 1

    if sim["bronze"] is not None:
        bronze_counts[sim["bronze"]] += 1

    if sim["fourth"] is not None:
        fourth_counts[sim["fourth"]] += 1

    final_pair_counts[sim["final_pair"]] += 1

    for t in sim["round32_teams"]:
        round32_counts[t] += 1

    for g, t in sim["group_winners"].items():
        group_winner_counts[t] += 1

    for t in sim["best_thirds"]:
        best_third_counts[t] += 1

    for t, status in sim["finish_status"].items():
        finish_counts[t][status] += 1

    for t, opp in sim["exit_opponent"].items():
        if opp is not None:
            exit_vs_counts[t][opp] += 1

    if i % max(1, N_SIMS // 10) == 0:
        elapsed = time.time() - start
        print(f"{i:,}/{N_SIMS:,} Simulationen fertig | {elapsed:.1f}s | Cache: {len(dist_cache)} Matchups")

elapsed = time.time() - start
print(f"\nFertig: {N_SIMS:,} Simulationen in {elapsed:.1f}s")
print("Unique predicted matchups in cache:", len(dist_cache))

# 10. Ergebnis-Tabellen

def pct(x):
    return round(100 * x / N_SIMS, 3)

title_rows = []
for team in all_teams:
    title_rows.append({
        "team": team,
        "gruppe": team_to_group[team],
        "weltmeister_%": pct(title_counts[team]),
        "finale_%": pct(title_counts[team] + silver_counts[team]),
        "halbfinale_medal_zone_%": pct(
            title_counts[team]
            + silver_counts[team]
            + bronze_counts[team]
            + fourth_counts[team]
        ),
        "16telfinale_%": pct(round32_counts[team]),
        "gruppensieger_%": pct(group_winner_counts[team]),
        "bester_dritter_%": pct(best_third_counts[team]),
    })

title_board = (
    pl.DataFrame(title_rows)
    .sort(["weltmeister_%", "finale_%", "16telfinale_%"], descending=True)
)

likely_round32_board = (
    title_board
    .sort(["16telfinale_%", "weltmeister_%"], descending=True)
    .head(32)
)

medal_rows = []
for team in all_teams:
    medal_rows.append({
        "team": team,
        "gold_%": pct(title_counts[team]),
        "silber_%": pct(silver_counts[team]),
        "bronze_%": pct(bronze_counts[team]),
        "vierter_%": pct(fourth_counts[team]),
    })

medal_board = (
    pl.DataFrame(medal_rows)
    .filter(
        (pl.col("gold_%") > 0)
        | (pl.col("silber_%") > 0)
        | (pl.col("bronze_%") > 0)
        | (pl.col("vierter_%") > 0)
    )
    .sort(["gold_%", "silber_%", "bronze_%"], descending=True)
)

final_pair_rows = []
for pair, c in final_pair_counts.most_common(20):
    final_pair_rows.append({
        "finale": f"{pair[0]} vs {pair[1]}",
        "wahrscheinlichkeit_%": pct(c),
    })

final_pair_board = pl.DataFrame(final_pair_rows)

def team_finish_board(team):
    rows = []
    for status, c in finish_counts[team].most_common():
        rows.append({
            "team": team,
            "finish": status,
            "wahrscheinlichkeit_%": pct(c),
        })

    return pl.DataFrame(
        rows,
        schema={
            "team": pl.String,
            "finish": pl.String,
            "wahrscheinlichkeit_%": pl.Float64,
        }
    )

def team_exit_vs_board(team, top_n=12):
    rows = []
    for opp, c in exit_vs_counts[team].most_common(top_n):
        rows.append({
            "team": team,
            "verliert_gegen": opp,
            "wahrscheinlichkeit_%": pct(c),
        })

    return pl.DataFrame(
        rows,
        schema={
            "team": pl.String,
            "verliert_gegen": pl.String,
            "wahrscheinlichkeit_%": pl.Float64,
        }
    )

def most_likely_exit_sentence(team):
    fb = team_finish_board(team)

    if fb.height == 0:
        return f"{team}: keine Finish-Daten gefunden."

    top = fb.row(0, named=True)

    if top["finish"] == "Champion":
        return f"{team}: häufigstes Ergebnis ist Weltmeister ({top['wahrscheinlichkeit_%']}%)."

    ev = team_exit_vs_board(team, top_n=1)

    if ev.height > 0:
        opp = ev.row(0, named=True)
        return (
            f"{team}: häufigstes Finish = {top['finish']} ({top['wahrscheinlichkeit_%']}%). "
            f"Häufigster Exit-Gegner: {opp['verliert_gegen']} ({opp['wahrscheinlichkeit_%']}%)."
        )

    return f"{team}: häufigstes Finish = {top['finish']} ({top['wahrscheinlichkeit_%']}%)."

# 11. Ausgaben

print("\n=== Titelwahrscheinlichkeiten Top 20 ===")
print(title_board.head(20))

print("\n=== Wahrscheinlichste 32 16telfinalisten + Titelchance ===")
print(likely_round32_board)

print("\n=== Medaillenwahrscheinlichkeiten ===")
print(medal_board.head(25))

print("\n=== Häufigste Finals ===")
print(final_pair_board)

print("\n=== USA: Wann fliegen sie raus? ===")
print(team_finish_board("USA"))
print("\nUSA häufigste Exit-Gegner:")
print(team_exit_vs_board("USA"))

print("\n=== Deutschland: Rundenverteilung ===")
print(team_finish_board("Germany"))
print("\nDeutschland häufigste Exit-Gegner:")
print(team_exit_vs_board("Germany"))

print("\n=== Südkorea: Rundenverteilung ===")
print(team_finish_board("South Korea"))
print("\nSüdkorea häufigste Exit-Gegner:")
print(team_exit_vs_board("South Korea"))

print("\n=== Kurzinterpretation ===")
print(most_likely_exit_sentence("USA"))
print(most_likely_exit_sentence("Germany"))
print(most_likely_exit_sentence("South Korea"))


## 32. Aggregierter Turnierbaum


In [ ]:
# Zelle 33: Code
# Was diese Zelle macht:
# Aggregiert aus vielen Simulationen einen Turnierbaum mit häufigsten Matchups, Scores und Gewinnern pro Spiel.

import re
import time
import numpy as np
import polars as pl
from collections import Counter, defaultdict

# Aggregierter Monte-Carlo-Turnierbaum mit exakten Scores

N_TREE_SIMS = 50_000
TREE_RNG_SEED = 123

# Wichtig: simulate_match/rank_group_rows nutzen global rng.
rng = np.random.default_rng(TREE_RNG_SEED)

pl.Config.set_tbl_cols(40)
pl.Config.set_tbl_rows(140)
pl.Config.set_tbl_width_chars(420)
pl.Config.set_fmt_str_lengths(220)

required_names = [
    "all_teams",
    "group_matches",
    "ko_matches",
    "team_id_to_name",
    "team_to_group",
    "stage_id_to_name",
    "city_id_to_country",
    "city_id_to_city",
    "parse_match_date",
    "simulate_match",
    "empty_table_row",
    "add_group_result",
    "rank_group_rows",
    "allocate_third_place_slots",
    "resolve_slot",
]

missing = [x for x in required_names if x not in globals()]
if missing:
    raise RuntimeError(
        "Diese Zelle braucht erst die große Monte-Carlo-Zelle davor. "
        f"Fehlt aktuell: {missing}"
    )

def pct_tree(x):
    return round(100 * x / N_TREE_SIMS, 3)

def split_match_label_tree(label):
    return [x.strip() for x in re.split(r"\s+vs\s+", str(label))]

def result_winner_label(res):
    if res["winner"] is None:
        return "Draw"
    return res["winner"]

def pretty_result(score, winner):
    if winner == "Draw":
        return f"{score} Remis"
    return f"{winner} gewinnt {score}"

def simulate_tournament_once_with_records():
    table = {team: empty_table_row(team) for team in all_teams}
    group_records = []

    # Gruppenphase
    for m in group_matches:
        mn = int(m["match_number"])
        home = team_id_to_name[int(m["home_team_id"])]
        away = team_id_to_name[int(m["away_team_id"])]
        d = parse_match_date(m["kickoff_at"])
        country = city_id_to_country[int(m["city_id"])]
        city = city_id_to_city[int(m["city_id"])]
        stage = stage_id_to_name[int(m["stage_id"])]

        fixed = get_played_match_result(mn, home, away) if "get_played_match_result" in globals() else None
        if fixed is not None:
            res = played_result_as_res(fixed, home, away, d, country, knockout=False, distribution_func=predict_match_distribution)
        else:
            res = simulate_match(home, away, d, country, knockout=False)

        add_group_result(table, home, away, res["home_score"], res["away_score"])

        winner = result_winner_label(res)

        group_records.append({
            "match_number": mn,
            "stage": stage,
            "label": str(m["match_label"]),
            "date": d,
            "city": city,
            "country": country,
            "home": home,
            "away": away,
            "score": res["score"],
            "winner": winner,
            "home_xg": res["home_xg"],
            "away_xg": res["away_xg"],
            "status": res.get("status", "Prognose"),
        })

    # Gruppentabellen / Qualifikation
    group_rankings = {}
    qualifiers = {}

    for g in sorted(set(team_to_group.values())):
        rows = [dict(v) for v in table.values() if v["group"] == g]
        ranked = rank_group_rows(rows)
        group_rankings[g] = ranked

        qualifiers[f"1{g}"] = ranked[0]["team"]
        qualifiers[f"2{g}"] = ranked[1]["team"]

    thirds = [group_rankings[g][2] for g in sorted(group_rankings.keys())]
    best_thirds = rank_group_rows(thirds)[:8]

    third_assignment = allocate_third_place_slots(best_thirds)
    qualifiers.update(third_assignment)

    round32_teams = set()
    for key, team in qualifiers.items():
        if re.fullmatch(r"[123][A-L]+", key):
            round32_teams.add(team)

    # KO-Phase
    match_results = {}
    ko_records = []

    for m in ko_matches:
        mn = int(m["match_number"])
        stage = stage_id_to_name[int(m["stage_id"])]
        label = str(m["match_label"])
        parts = split_match_label_tree(label)

        if len(parts) != 2:
            raise RuntimeError(f"Kann match_label nicht splitten: {label}")

        home = resolve_slot(parts[0], qualifiers, match_results)
        away = resolve_slot(parts[1], qualifiers, match_results)

        d = parse_match_date(m["kickoff_at"])
        country = city_id_to_country[int(m["city_id"])]
        city = city_id_to_city[int(m["city_id"])]

        res = simulate_match(home, away, d, country, knockout=True)
        match_results[mn] = res

        winner = result_winner_label(res)

        ko_records.append({
            "match_number": mn,
            "stage": stage,
            "label": label,
            "date": d,
            "city": city,
            "country": country,
            "home": home,
            "away": away,
            "score": res["score"],
            "winner": winner,
            "loser": res["loser"],
            "home_xg": res["home_xg"],
            "away_xg": res["away_xg"],
        })

    final_rec = next((r for r in ko_records if str(r["stage"]).lower() == "final"), None)
    third_rec = next((r for r in ko_records if "third" in str(r["stage"]).lower()), None)

    gold = final_rec["winner"]
    silver = final_rec["loser"]
    bronze = third_rec["winner"] if third_rec else None
    fourth = third_rec["loser"] if third_rec else None

    return {
        "group_records": group_records,
        "ko_records": ko_records,
        "round32_teams": round32_teams,
        "gold": gold,
        "silver": silver,
        "bronze": bronze,
        "fourth": fourth,
    }

# Simulationen aggregieren

match_meta = {}
matchup_counts = defaultdict(Counter)
result_counts = defaultdict(Counter)
winner_counts = defaultdict(Counter)
score_counts = defaultdict(Counter)

title_counts_tree = Counter()
silver_counts_tree = Counter()
bronze_counts_tree = Counter()
fourth_counts_tree = Counter()
round32_counts_tree = Counter()

start = time.time()

for i in range(1, N_TREE_SIMS + 1):
    sim = simulate_tournament_once_with_records()

    title_counts_tree[sim["gold"]] += 1
    silver_counts_tree[sim["silver"]] += 1
    if sim["bronze"] is not None:
        bronze_counts_tree[sim["bronze"]] += 1
    if sim["fourth"] is not None:
        fourth_counts_tree[sim["fourth"]] += 1

    for t in sim["round32_teams"]:
        round32_counts_tree[t] += 1

    for rec in sim["group_records"] + sim["ko_records"]:
        mn = rec["match_number"]
        home = rec["home"]
        away = rec["away"]
        score = rec["score"]
        winner = rec["winner"]

        match_meta[mn] = {
            "match_number": mn,
            "stage": rec["stage"],
            "label": rec["label"],
            "date": rec["date"],
            "city": rec["city"],
            "country": rec["country"],
        }

        matchup_counts[mn][(home, away)] += 1
        result_counts[mn][(home, away, score, winner)] += 1
        winner_counts[mn][winner] += 1
        score_counts[mn][score] += 1

    if i % max(1, N_TREE_SIMS // 10) == 0:
        elapsed = time.time() - start
        print(f"{i:,}/{N_TREE_SIMS:,} Simulationen aggregiert | {elapsed:.1f}s")

print(f"\nFertig in {time.time() - start:.1f}s")

# Aggregierten Turnierbaum bauen

tree_rows = []

for mn in sorted(match_meta.keys()):
    meta = match_meta[mn]

    top_matchup, top_matchup_n = matchup_counts[mn].most_common(1)[0]
    top_home, top_away = top_matchup

    # Häufigstes kohärentes Ergebnis innerhalb des häufigsten Matchups.
    result_counter_for_top_matchup = Counter()
    for (h, a, score, winner), c in result_counts[mn].items():
        if h == top_home and a == top_away:
            result_counter_for_top_matchup[(score, winner)] += c

    (top_score_for_matchup, top_result_winner), top_result_n = result_counter_for_top_matchup.most_common(1)[0]

    # Häufigster Gewinner insgesamt, unabhängig vom Matchup.
    top_winner, top_winner_n = winner_counts[mn].most_common(1)[0]

    # Häufigster Score insgesamt, unabhängig vom Matchup.
    top_score_overall, top_score_overall_n = score_counts[mn].most_common(1)[0]

    tree_rows.append({
        "match": mn,
        "stage": meta["stage"],
        "label": meta["label"],
        "date": meta["date"],
        "city": meta["city"],
        "country": meta["country"],

        "häufigster_matchup": f"{top_home} vs {top_away}",
        "matchup_%": round(100 * top_matchup_n / N_TREE_SIMS, 3),

        "häufigstes_resultat": pretty_result(top_score_for_matchup, top_result_winner),
        "resultat_%_wenn_matchup": round(100 * top_result_n / top_matchup_n, 3),
        "resultat_%_gesamt": round(100 * top_result_n / N_TREE_SIMS, 3),

        "häufigster_score_gesamt": top_score_overall,
        "score_gesamt_%": round(100 * top_score_overall_n / N_TREE_SIMS, 3),

        "häufigster_winner_gesamt": top_winner,
        "winner_gesamt_%": round(100 * top_winner_n / N_TREE_SIMS, 3),
    })

full_tree_agg = pl.DataFrame(tree_rows).sort("match")

group_tree_agg = full_tree_agg.filter(pl.col("stage").str.contains("Group"))
ko_tree_agg = full_tree_agg.filter(~pl.col("stage").str.contains("Group"))

# 16telfinalisten + Titelchance aus derselben Aggregation

round32_title_rows = []

for team in all_teams:
    round32_title_rows.append({
        "team": team,
        "gruppe": team_to_group[team],
        "16telfinale_%": pct_tree(round32_counts_tree[team]),
        "weltmeister_%": pct_tree(title_counts_tree[team]),
        "finale_%": pct_tree(title_counts_tree[team] + silver_counts_tree[team]),
        "halbfinale_medal_zone_%": pct_tree(
            title_counts_tree[team]
            + silver_counts_tree[team]
            + bronze_counts_tree[team]
            + fourth_counts_tree[team]
        ),
    })

round32_title_board = (
    pl.DataFrame(round32_title_rows)
    .sort(["16telfinale_%", "weltmeister_%"], descending=True)
)

likely_16telfinalisten_title = round32_title_board.head(32)

# Match-Detail-Helfer

def show_match_details(match_number, top_n=12):
    mn = int(match_number)

    if mn not in match_meta:
        print(f"Match {mn} nicht gefunden.")
        return

    meta = match_meta[mn]
    print(f"Match {mn} | {meta['stage']} | {meta['label']} | {meta['date']} | {meta['city']}, {meta['country']}")

    matchup_rows = []
    for (home, away), c in matchup_counts[mn].most_common(top_n):
        matchup_rows.append({
            "matchup": f"{home} vs {away}",
            "wahrscheinlichkeit_%": round(100 * c / N_TREE_SIMS, 3),
        })

    result_rows = []
    for (home, away, score, winner), c in result_counts[mn].most_common(top_n):
        result_rows.append({
            "matchup": f"{home} vs {away}",
            "resultat": pretty_result(score, winner),
            "wahrscheinlichkeit_%": round(100 * c / N_TREE_SIMS, 3),
        })

    print("\nHäufigste Matchups:")
    print(pl.DataFrame(matchup_rows))

    print("\nHäufigste exakte Resultate:")
    print(pl.DataFrame(result_rows))

# Ausgaben

print("\n=== Alle Gruppenspiele aggregiert ===")
print(
    group_tree_agg.select([
        "match",
        "label",
        "date",
        "city",
        "häufigster_matchup",
        "häufigstes_resultat",
        "resultat_%_wenn_matchup",
        "häufigster_winner_gesamt",
        "winner_gesamt_%",
    ])
)

print("\n=== Aggregierter KO-Turnierbaum ===")

stage_order = [
    "Round of 32",
    "Round of 16",
    "Quarterfinals",
    "Semifinals",
    "Third Place Playoff",
    "Final",
]

for stage in stage_order:
    part = ko_tree_agg.filter(pl.col("stage") == stage)
    if part.height == 0:
        continue

    print(f"\n--- {stage} ---")
    print(
        part.select([
            "match",
            "label",
            "date",
            "city",
            "häufigster_matchup",
            "matchup_%",
            "häufigstes_resultat",
            "resultat_%_wenn_matchup",
            "resultat_%_gesamt",
            "häufigster_winner_gesamt",
            "winner_gesamt_%",
        ])
    )

print("\n=== Wahrscheinlichste 32 16telfinalisten + WM-Titelchance ===")
print(likely_16telfinalisten_title)

print("\n=== Titelwahrscheinlichkeiten Top 20 ===")
print(
    round32_title_board
    .sort(["weltmeister_%", "finale_%"], descending=True)
    .head(20)
)

print("\nTipp: Details zu einem einzelnen Match anzeigen, z.B. Finale:")
print("show_match_details(104)")


## 34. Rundenwahrscheinlichkeiten und Exit-Details


## 35. Aggregierter Turnierbaum

Aggregiert aus vielen Simulationen einen Turnierbaum mit häufigsten Matchups, Scores und Gewinnern pro Spiel.


In [ ]:
# Zelle 36: Code
# Was diese Zelle macht:
# Gibt für ein Team aus, wie oft es jede KO-Runde erreicht, dort ausscheidet,
# das Finale erreicht und Weltmeister wird.

import polars as pl
from collections import Counter, defaultdict

TEAM_TO_ANALYZE = "Germany"

required = ["result_counts", "match_meta", "N_TREE_SIMS"]
missing = [x for x in required if x not in globals()]
if missing:
    raise RuntimeError(f"Bitte zuerst die Monte-Carlo-Aggregation laufen lassen. Fehlt: {missing}")

STAGE_ORDER = [
    "Round of 32",
    "Round of 16",
    "Quarterfinals",
    "Semifinals",
    "Final",
]

def pct(x):
    return round(100 * x / N_TREE_SIMS, 3)

def stage_key(stage):
    s = str(stage).lower()
    if "round of 32" in s:
        return "Round of 32"
    if "round of 16" in s:
        return "Round of 16"
    if "quarter" in s:
        return "Quarterfinals"
    if "semi" in s:
        return "Semifinals"
    if s == "final" or " final" in s:
        return "Final"
    return None

reached_counts = Counter()
winner_counts_by_stage = Counter()
lost_counts_by_stage = Counter()

for match_no, counter in result_counts.items():
    stage = stage_key(match_meta[match_no]["stage"])
    if stage is None:
        continue

    for (home, away, score, winner), c in counter.items():
        if TEAM_TO_ANALYZE in (home, away):
            reached_counts[stage] += c

            if winner == TEAM_TO_ANALYZE:
                winner_counts_by_stage[stage] += c
            else:
                lost_counts_by_stage[stage] += c

champion_count = winner_counts_by_stage["Final"]
final_lost_count = lost_counts_by_stage["Final"]

# Gruppenphase raus = nicht Round of 32 erreicht
group_exit_count = N_TREE_SIMS - reached_counts["Round of 32"]

rows = []

rows.append({
    "phase": "Gruppenphase überstehen / 16-telfinale erreichen",
    "erreicht_%": pct(reached_counts["Round of 32"]),
    "scheidet_hier_aus_%": pct(group_exit_count),
})

for stage in STAGE_ORDER:
    rows.append({
        "phase": stage,
        "erreicht_%": pct(reached_counts[stage]),
        "scheidet_hier_aus_%": pct(lost_counts_by_stage[stage]),
    })

rows.append({
    "phase": "Weltmeister",
    "erreicht_%": pct(champion_count),
    "scheidet_hier_aus_%": 0.0,
})

team_path_summary = pl.DataFrame(rows)

print(f"=== Turnierchancen: {TEAM_TO_ANALYZE} ===")
print(team_path_summary)

print("\nKurzfassung:")
print(f"16-telfinale erreichen: {pct(reached_counts['Round of 32'])}%")
print(f"Achtelfinale erreichen: {pct(reached_counts['Round of 16'])}%")
print(f"Viertelfinale erreichen: {pct(reached_counts['Quarterfinals'])}%")
print(f"Halbfinale erreichen: {pct(reached_counts['Semifinals'])}%")
print(f"Finale erreichen: {pct(reached_counts['Final'])}%")
print(f"Weltmeister werden: {pct(champion_count)}%")


## 37. Exakte KO-Ausscheidungen

Analysiert für Deutschland und Südkorea, in welcher KO-Runde und mit welchem konkreten Endstand sie am häufigsten ausscheiden.


In [ ]:
# Zelle 38: Code
# Was diese Zelle macht:
# Analysiert für Deutschland und Südkorea, in welcher KO-Runde und mit welchem konkreten Endstand sie am häufigsten ausscheiden.

import re
import polars as pl
from collections import Counter, defaultdict

pl.Config.set_tbl_cols(30)
pl.Config.set_tbl_rows(80)
pl.Config.set_tbl_width_chars(340)
pl.Config.set_fmt_str_lengths(180)

TARGET_TEAMS = ["Germany", "South Korea"]

required = ["result_counts", "match_meta", "N_TREE_SIMS"]
missing = [x for x in required if x not in globals()]
if missing:
    raise RuntimeError(
        "Diese Zelle muss nach der aggregierten Turnierbaum-Zelle laufen. "
        f"Fehlt aktuell: {missing}"
    )

def parse_score_team_view(score, home, away, team):
    """
    Wandelt Home/Away-Score in Team-Sicht um.
    Beispiel:
    home=France, away=Germany, score=2:1, team=Germany -> 1:2
    """
    score_str = str(score)
    pen = "n.E." in score_str

    base = score_str.replace(" n.E.", "").strip()
    hg, ag = map(int, base.split(":"))

    if team == home:
        tg, og = hg, ag
    elif team == away:
        tg, og = ag, hg
    else:
        raise ValueError(f"{team} ist nicht in Match: {home} vs {away}")

    out = f"{tg}:{og}"
    if pen:
        out += " n.E."
    return out

def is_title_exit_stage(stage):
    s = str(stage).lower()
    if "group" in s:
        return False
    if "third" in s:
        return False
    return True

exit_exact_counts = {team: Counter() for team in TARGET_TEAMS}
exit_score_counts = {team: Counter() for team in TARGET_TEAMS}
exit_stage_counts = {team: Counter() for team in TARGET_TEAMS}
exit_stage_score_counts = {team: Counter() for team in TARGET_TEAMS}

for mn, counter in result_counts.items():
    meta = match_meta[mn]
    stage = meta["stage"]

    if not is_title_exit_stage(stage):
        continue

    for (home, away, score, winner), c in counter.items():
        for team in TARGET_TEAMS:
            if team not in (home, away):
                continue

            # In KO-Spielen heißt: Team war dabei und hat nicht gewonnen -> Titel-Aus.
            if winner == team:
                continue

            opponent = away if team == home else home
            team_score = parse_score_team_view(score, home, away, team)

            key = (stage, opponent, team_score)
            exit_exact_counts[team][key] += c
            exit_score_counts[team][team_score] += c
            exit_stage_counts[team][stage] += c
            exit_stage_score_counts[team][(stage, team_score)] += c

def pct(x, denom=N_TREE_SIMS):
    return round(100 * x / denom, 3) if denom else 0.0

overview_rows = []
exact_rows = []
score_rows = []
stage_score_rows = []

for team in TARGET_TEAMS:
    ko_exit_n = sum(exit_exact_counts[team].values())

    round32_n = round32_counts_tree[team] if "round32_counts_tree" in globals() else None
    champion_n = title_counts_tree[team] if "title_counts_tree" in globals() else None
    group_stage_n = N_TREE_SIMS - round32_n if round32_n is not None else None

    overview_rows.append({
        "team": team,
        "gruppenphase_raus_%": pct(group_stage_n) if group_stage_n is not None else None,
        "ko_ausscheiden_mit_score_%": pct(ko_exit_n),
        "weltmeister_%": pct(champion_n) if champion_n is not None else None,
        "ko_exit_count": ko_exit_n,
    })

    for (stage, opponent, team_score), c in exit_exact_counts[team].most_common(20):
        exact_rows.append({
            "team": team,
            "stage": stage,
            "gegner": opponent,
            "team_score": team_score,
            "lesart": f"{team} verliert {team_score} gegen {opponent}",
            "gesamt_%": pct(c),
            "wenn_ko_exit_%": round(100 * c / ko_exit_n, 3) if ko_exit_n else 0.0,
        })

    for team_score, c in exit_score_counts[team].most_common(12):
        score_rows.append({
            "team": team,
            "team_score": team_score,
            "gesamt_%": pct(c),
            "wenn_ko_exit_%": round(100 * c / ko_exit_n, 3) if ko_exit_n else 0.0,
        })

    for (stage, team_score), c in exit_stage_score_counts[team].most_common(20):
        stage_score_rows.append({
            "team": team,
            "stage": stage,
            "team_score": team_score,
            "gesamt_%": pct(c),
            "wenn_ko_exit_%": round(100 * c / ko_exit_n, 3) if ko_exit_n else 0.0,
        })

exit_overview = pl.DataFrame(overview_rows)
exit_exact_top = pl.DataFrame(exact_rows)
exit_score_top = pl.DataFrame(score_rows)
exit_stage_score_top = pl.DataFrame(stage_score_rows)

print("=== Übersicht ===")
print(exit_overview)

print("\n=== Häufigste konkrete KO-Ausscheidungen mit Gegner + Endstand ===")
print(exit_exact_top)

print("\n=== Häufigste reine Ausscheidungs-Endstände, egal gegen wen ===")
print(exit_score_top)

print("\n=== Häufigste Ausscheidungs-Endstände je Runde ===")
print(exit_stage_score_top)
